# Phase 3 · Rhetorical Role Prediction (RR)

**Task:** assign one of 13 rhetorical roles (Fact, Issue, ArgumentPetitioner, …, RulingByPresentCourt, None) to **every sentence** of a judgment. There are two domains, **Corporate Law (CL)** and **IT**, with separate train/dev/test files. **Metric: macro-F1.** Output: `{doc_id: [label per sentence]}`, in order and with exactly one label per sentence.

**Requirements (§9):** start from the Phase 1 models and use all four. At least 6 substantially different experiments: at least 5 different fine-tuning / PEFT techniques from the list (LoRA, QLoRA, AdaLoRA, rsLoRA, **LoRMA (compulsory)**, prefix tuning, prompt tuning, IA³) plus at least 1 **knowledge distillation** experiment. Study **sentence-only vs. document context**, compare **CL vs. IT**, and analyse confusions between closely related roles.

**P3-0:** restore the Phase 1 checkpoints if needed, download the RR data, and profile it (sizes, lengths in our tokens, label distribution, label persistence and position). These statistics decide the design.

In [1]:
# ============================================================================================
# P3-0 — Setup + data profile for Phase 3 (Rhetorical Role prediction)
#   CPU only. Restores Phase-1 models/code from HF only if they are missing locally.
# ============================================================================================
import os, sys, json, glob
import numpy as np
from collections import Counter, defaultdict
from huggingface_hub import HfApi, snapshot_download
from kaggle_secrets import UserSecretsClient

WORK  = "/kaggle/working"
TOKEN = UserSecretsClient().get_secret("HF_WRITE_TOKEN")           # HF_TOKEN gave 401 on this account
MODELS = ["m1_dense_100m", "m2_mla_moe", "m3_encdec", "m4_copy"]    # the four Phase-1 models

# ---- 1. make sure code, tokenizer and the four Phase-1 checkpoints exist (restore if not) ----
need = [f"{WORK}/src/train.py", f"{WORK}/src/p2_model.py", f"{WORK}/tokenizer/hf_a1_bpe_v2_32k.json"] + \
       [f"{WORK}/checkpoints/{m}/best.pt" for m in MODELS]
missing = [p for p in need if not os.path.exists(p)]
if missing:
    print("restoring from HF:", [os.path.relpath(p, WORK) for p in missing])
    me = HfApi(token=TOKEN).whoami()["name"]
    snapshot_download(f"{me}/cs781-phase1-work", local_dir=WORK, token=TOKEN,
                      allow_patterns=["src/**", "tokenizer/**"] +
                                     [f"checkpoints/{m}/best.pt" for m in MODELS] +
                                     [f"checkpoints/{m}/config.json" for m in MODELS])
print("phase-1 assets OK:", all(os.path.exists(p) for p in need))
sys.path.insert(0, f"{WORK}/src")

# ---- 2. download the Phase-3 data ----------------------------------------------------------------
snapshot_download("Exploration-Lab/CS-781-Capstone", repo_type="dataset", token=TOKEN,
                  allow_patterns=["Phase_3_RR/**"], local_dir=f"{WORK}/data_p3")
files = sorted(glob.glob(f"{WORK}/data_p3/**/*.json", recursive=True))
print("\nfiles:"); [print(f"   {os.path.relpath(f, WORK)}  {os.path.getsize(f)/1e6:.2f} MB") for f in files]
F = {os.path.basename(f).replace(".json", ""): f for f in files}

def load_docs(path):
    """Accept a list of docs or a dict id -> doc; always return a list of {id, text, labels?}."""
    obj = json.load(open(path))
    if isinstance(obj, dict):
        obj = [{"id": k, **v} if isinstance(v, dict) else {"id": k, "text": v} for k, v in obj.items()]
    return obj

VOCAB = json.load(open(F["label_vocab"]))
VOCAB = list(VOCAB) if not isinstance(VOCAB, dict) else list(VOCAB.keys() if all(isinstance(v, int) for v in VOCAB.values()) else VOCAB.values())
print(f"\nlabel_vocab ({len(VOCAB)}):", VOCAB)

# ---- 3. tokenizer (same normalisation as pre-training) -----------------------------------------
from tokenizers import Tokenizer
from common import normalise
tk = Tokenizer.from_file(f"{WORK}/tokenizer/hf_a1_bpe_v2_32k.json")
ntok = lambda s: len(tk.encode(normalise(s), add_special_tokens=False).ids)

# ---- 4. profile every split --------------------------------------------------------------------
D = {}
first = True
for name in ["CL_train", "CL_dev", "CL_test", "IT_train", "IT_dev", "IT_test"]:
    if name not in F:
        print(f"\n{name}: NOT FOUND"); continue
    docs = D[name] = load_docs(F[name])
    if first:
        print("\nexample doc keys:", list(docs[0].keys()))
        print("example:", {k: (v[:2] if isinstance(v, list) else v) for k, v in docs[0].items()}); first = False
    ns = [len(d["text"]) for d in docs]
    sl = [ntok(s) for d in docs for s in d["text"]]
    dl = [sum(ntok(s) for s in d["text"]) for d in docs]
    print(f"\n=== {name}: {len(docs)} docs | {sum(ns)} sentences | sents/doc median {int(np.median(ns))} max {max(ns)}")
    print(f"    tokens/sentence: median {int(np.median(sl))} | p95 {int(np.percentile(sl, 95))} | max {max(sl)}")
    print(f"    tokens/doc:      median {int(np.median(dl))} | p90 {int(np.percentile(dl, 90))} | max {max(dl)}"
          f" | docs > 1024 tok: {sum(x > 1024 for x in dl)}")
    if "labels" in docs[0]:
        assert all(len(d["text"]) == len(d["labels"]) for d in docs), "text/labels length mismatch!"
        bad = {l for d in docs for l in d["labels"]} - set(VOCAB)
        if bad: print("    labels NOT in vocab:", bad)

# ---- 5. label distribution per domain (train+dev), persistence and position ----------------------
for dom in ["CL", "IT"]:
    docs = D.get(f"{dom}_train", []) + D.get(f"{dom}_dev", [])
    labs = [l for d in docs for l in d["labels"]]
    cnt = Counter(labs); n = len(labs)
    same = np.mean([a == b for d in docs for a, b in zip(d["labels"][:-1], d["labels"][1:])])
    pos = defaultdict(list)                                          # relative position 0..1 of each label
    for d in docs:
        L = len(d["labels"])
        for i, l in enumerate(d["labels"]):
            pos[l].append(i / max(1, L - 1))
    print(f"\n=== {dom} label distribution (train+dev, {n} sentences) | P(label = previous label) = {same:.2f}")
    for l in VOCAB:
        c = cnt.get(l, 0)
        p = f"{np.mean(pos[l]):.2f}" if pos[l] else "  - "
        print(f"   {l:<24}{c:>6}  {100*c/max(1,n):5.1f}%   mean position {p}")

phase-1 assets OK: True


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]


files:
   data_p3/Phase_3_RR/Test/CL_test.json  0.28 MB
   data_p3/Phase_3_RR/Test/IT_test.json  0.19 MB
   data_p3/Phase_3_RR/Train/CL_dev.json  0.34 MB
   data_p3/Phase_3_RR/Train/CL_train.json  2.46 MB
   data_p3/Phase_3_RR/Train/IT_dev.json  0.17 MB
   data_p3/Phase_3_RR/Train/IT_train.json  1.51 MB
   data_p3/Phase_3_RR/Train/label_vocab.json  0.00 MB

label_vocab (13): ['Fact', 'Issue', 'ArgumentPetitioner', 'ArgumentRespondent', 'Statute', 'Dissent', 'PrecedentReliedUpon', 'PrecedentNotReliedUpon', 'PrecedentOverruled', 'RulingByLowerCourt', 'RatioOfTheDecision', 'RulingByPresentCourt', 'None']

example doc keys: ['id', 'text', 'labels']
example: {'id': 'CCI_All_India_Distillers_Association_vs_Haldyn_Glass_GCO100028COM855733', 'text': ['Consequent upon the repeal of the MRTP Act this complaint has been received by transfer on 04.03.2010 from the office of DGIR, MRTP Commission under Section 66(6) of the Competition Act, 2002.', 'The facts as stated in the complaint, in brief, a

## P3-1 · Pipeline (`p3_rr.py`)

* **Input:** windows of consecutive sentences `[SEP] s1 [SEP] s2 …`, at most W = 512 tokens, about 50% overlap. `ctx="none"` puts one sentence per window (the sentence-only baseline).
* **Sentence representation:** mean of the sentence's hidden states (its [SEP] marker plus its tokens), plus a zero-initialised **relative-position embedding** (10 buckets).
* **Prediction:** linear head over 13 roles; a sentence's log-probs are averaged over every window containing it.
* **Domain priors:** a per-domain **label mask** (roles never seen in that domain's training data are never predicted) and an optional **transition Viterbi** (λ · log P(role_t | role_t−1), estimated from training data).
* **Loss:** class-weighted CE (inverse √frequency, capped at 10). Epoch selection by dev macro-F1 over CL+IT.
* **PEFT:** everything from Phase 2 (LoRA, LoRMA+/π, prompt, full) plus **rsLoRA** (α/√r scaling) and **IA³** (learned rescaling of keys, values and the FFN activation).

## P3-2 · Baselines: does context help?

M1 + LoRA (r = 8, q and v), bidirectional attention, joint CL+IT training. Three variants: **sentence-only**, **document context**, **document context + position**. Each is also decoded with transition Viterbi at λ ∈ {0, 0.5, 1}.

In [2]:
%%writefile /kaggle/working/src/p3_rr.py
"""
p3_rr.py -- Phase 3 (Rhetorical Role prediction) on top of ANY Phase-1 backbone.

Idea: a document is a sequence of sentences. We feed WINDOWS of consecutive sentences to the
backbone as one token stream  [SEP] s_1 [SEP] s_2 ... , pool the hidden states of each sentence
(its [SEP] marker + its tokens, mean) and classify the pooled vector into one of 13 roles.
  ctx="none" : every window holds ONE sentence           -> sentence-only model
  ctx="doc"  : windows of ~W tokens of neighbouring sentences, 50% overlap -> document context
               (a sentence's prediction = mean log-prob over the windows that contain it)
Optional extras:
  * relative-position embedding (10 buckets of i / n_sentences), zero-initialised
  * per-domain label mask (never predict a role that does not occur in that domain's training data)
  * label-transition Viterbi (roles come in runs: P(same as previous) = 0.94 CL / 0.75 IT)

PEFT methods: everything in p2_peft.apply_method (lora, lorma_plus, lorma_pi, prompt, full, ...)
plus two new ones for Phase 3:
  "rslora" : LoRA with rank-stabilised scaling alpha / sqrt(r) (Kalajdzievski, 2023)
  "ia3"    : IA^3 (Liu et al., 2022): learned vectors rescale keys, values and the FFN
             intermediate activation (input of the down-projection); all initialised to 1
"""
from __future__ import annotations

import contextlib
import math
import random
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import p2_model as pm
import p2_peft as pp
from common import normalise
from lm_common import SwiGLU

LABELS = ['Fact', 'Issue', 'ArgumentPetitioner', 'ArgumentRespondent', 'Statute', 'Dissent',
          'PrecedentReliedUpon', 'PrecedentNotReliedUpon', 'PrecedentOverruled', 'RulingByLowerCourt',
          'RatioOfTheDecision', 'RulingByPresentCourt', 'None']
L2I = {l: i for i, l in enumerate(LABELS)}
NL = len(LABELS)
SEP_ID, PAD_ID = 3, 0            # <|sep|> marks the start of every sentence; padding is masked out
MAX_SENT = 128                   # truncate very long sentences (p95 is ~90-107 tokens)
N_POS = 10                       # relative-position buckets


def _autocast(device):
    dev = str(device)
    return (torch.autocast("cuda", dtype=torch.float16) if dev.startswith("cuda")
            else contextlib.nullcontext())


# ----------------------------------------------------------------------------- data
def encode_docs(docs, tk, domain):
    """{key: {sents: [token ids], labs: [label ids or -100], dom, orig_id}}; key = 'DOM::id'."""
    out = {}
    for d in docs:
        sents = [tk.encode(normalise(s), add_special_tokens=False).ids[:MAX_SENT] for s in d["text"]]
        labs = [L2I[l] for l in d["labels"]] if "labels" in d else [-100] * len(sents)
        out[f"{domain}::{d['id']}"] = {"sents": sents, "labs": labs, "dom": domain, "orig_id": d["id"]}
    return out


def sent_windows(sents, W=512, ctx="doc"):
    """Sentence ranges [a, b). ctx='none': one sentence per window. ctx='doc': greedy windows of at
    most W tokens; the next window starts at the sentence where the previous one reached half its
    tokens (~50% overlap), so every sentence is seen with left AND right context."""
    n = len(sents)
    if ctx == "none":
        return [(i, i + 1) for i in range(n)]
    L = [len(s) + 1 for s in sents]                                  # +1 for the [SEP] marker
    wins, a = [], 0
    while a < n:
        b, tot = a, 0
        while b < n and (tot + L[b] <= W or b == a):
            tot += L[b]; b += 1
        wins.append((a, b))
        if b >= n:
            break
        a2, t = a, 0
        while a2 < b and t + L[a2] <= tot / 2:
            t += L[a2]; a2 += 1
        while a2 < b and sum(L[a2:b]) + L[b] > W:                     # next window must add new sentences
            a2 += 1
        a = max(a2, a + 1)
    return wins


def make_examples(enc, keys, ctx="doc", W=512):
    ex = []
    for k in keys:
        e = enc[k]; n = len(e["sents"])
        for a, b in sent_windows(e["sents"], W, ctx):
            ids, seg = [], []
            for j, i in enumerate(range(a, b)):
                s = [SEP_ID] + e["sents"][i]
                ids += s; seg += [j] * len(s)
            ex.append({"ids": ids, "seg": seg, "labs": e["labs"][a:b], "key": k, "sidx": list(range(a, b)),
                       "pos": [min(N_POS - 1, int(N_POS * i / max(1, n))) for i in range(a, b)]})
    return ex


def collate(batch, device):
    B, L = len(batch), max(len(x["ids"]) for x in batch)
    ids = torch.full((B, L), PAD_ID, dtype=torch.long)
    seg = torch.full((B, L), -1, dtype=torch.long)
    labs, pos, off, o = [], [], [], 0
    for b, x in enumerate(batch):
        n = len(x["ids"])
        ids[b, :n] = torch.tensor(x["ids"]); seg[b, :n] = torch.tensor(x["seg"])
        labs += x["labs"]; pos += x["pos"]; off.append(o); o += len(x["labs"])
    return (ids.to(device), seg.to(device), torch.tensor(off, device=device),
            torch.tensor(labs, device=device), torch.tensor(pos, device=device))


# ----------------------------------------------------------------------------- model
class RRModel(nn.Module):
    """Backbone (TokenTagger from Phase 2) -> mean-pool per sentence (+ position) -> role logits."""
    def __init__(self, tagger, use_pos=True, dropout=0.1):
        super().__init__()
        self.tagger, self.use_pos = tagger, use_pos
        d = tagger.backbone.cfg.d_model
        self.drop = nn.Dropout(dropout)
        self.pos = nn.Embedding(N_POS, d)
        nn.init.zeros_(self.pos.weight)                               # starts as "no position info"

    def forward(self, ids, seg, off, pos):
        tg = self.tagger
        ctx = pm.full_attention() if (tg.bidirectional and tg.arch != "encdec") else contextlib.nullcontext()
        with ctx:
            h = tg.hidden(ids)
        head = tg.head
        if isinstance(head, pp.DropPrefix):                           # prompt tuning: drop soft prompt
            h, head = h[:, head.n:], head.head
        h = h.float()
        valid = seg >= 0
        gid = (seg + off[:, None])[valid]                             # global sentence index per token
        n = pos.numel()
        pooled = torch.zeros(n, h.size(-1), device=h.device).index_add_(0, gid, h[valid])
        cnt = torch.zeros(n, device=h.device).index_add_(0, gid, torch.ones_like(gid, dtype=torch.float))
        pooled = pooled / cnt.clamp(min=1)[:, None]
        if self.use_pos:
            pooled = pooled + self.pos(pos)
        return head(self.drop(pooled))                                # (n_sentences, NL)


class RSLoRALinear(pp.LoRALinear):
    """LoRA with rank-stabilised scaling alpha / sqrt(r) instead of alpha / r."""
    def __init__(self, base, r=16, alpha=16):
        super().__init__(base, r, alpha)
        self.s = alpha / math.sqrt(r)


class IA3Out(nn.Module):
    """y = (W x) * l  (rescales keys / values)."""
    def __init__(self, base):
        super().__init__()
        self.base = base
        self.l = nn.Parameter(torch.ones(base.out_features))

    def forward(self, x):
        return self.base(x) * self.l


class IA3In(nn.Module):
    """y = W (x * l)  (rescales the FFN intermediate activation before the down-projection)."""
    def __init__(self, base):
        super().__init__()
        self.base = base
        self.l = nn.Parameter(torch.ones(base.in_features))

    def forward(self, x):
        return self.base(x * self.l)


def apply_method_p3(tagger, method, r=8, alpha=8, **kw):
    if method == "rslora":
        for p in tagger.backbone.parameters():
            p.requires_grad = False
        rr = kw.pop("rs_r", 16)
        pp._wrap_linears(tagger, lambda l: RSLoRALinear(l, rr, rr))
    elif method == "ia3":
        for p in tagger.backbone.parameters():
            p.requires_grad = False
        for layer in pp._layers(tagger):
            for mod in list(layer.modules()):
                for name in ("wk", "wv", "w_uk", "w_uv"):            # keys / values (dense + MLA)
                    lin = getattr(mod, name, None)
                    if isinstance(lin, nn.Linear):
                        setattr(mod, name, IA3Out(lin))
                if isinstance(mod, SwiGLU) and isinstance(mod.w2, nn.Linear):
                    mod.w2 = IA3In(mod.w2)                            # FFN intermediate activation
    else:
        pp.apply_method(tagger, method, r=r, alpha=alpha, **kw)
    for p in tagger.head.parameters():
        p.requires_grad = True
    return tagger


def build_rr(ckpt, method="lora", bidirectional=True, use_pos=True, **kw):
    tg = pm.build_tagger(ckpt, bidirectional=bidirectional)
    tg.head = nn.Linear(tg.backbone.cfg.d_model, NL)
    nn.init.normal_(tg.head.weight, 0.0, 0.02); nn.init.zeros_(tg.head.bias)
    apply_method_p3(tg, method, **kw)
    return RRModel(tg, use_pos=use_pos)


def save_trainable(model, path):
    """Only the parameters that were trained (PEFT runs stay a few MB)."""
    torch.save({n: p.detach().half().cpu() for n, p in model.named_parameters() if p.requires_grad}, path)


def load_trainable(model, path):
    sd = {k: v.float() for k, v in torch.load(path, map_location="cpu").items()}
    missing, unexpected = model.load_state_dict(sd, strict=False)
    assert not unexpected, unexpected
    return model


# ----------------------------------------------------------------------------- domain priors
def domain_stats(enc, keys):
    """Per domain: allowed-label mask (seen in training) and log transition matrix (add-1)."""
    allowed, trans = {}, {}
    for dom in sorted({enc[k]["dom"] for k in keys}):
        C = np.ones((NL, NL)); seen = np.zeros(NL, dtype=bool)
        for k in keys:
            if enc[k]["dom"] != dom:
                continue
            labs = enc[k]["labs"]
            for l in labs:
                seen[l] = True
            for a, b in zip(labs[:-1], labs[1:]):
                C[a, b] += 1
        allowed[dom], trans[dom] = seen, np.log(C / C.sum(1, keepdims=True))
    return allowed, trans


def viterbi(score, trans):
    n, K = score.shape
    dp, back = score[0].copy(), np.zeros((n, K), dtype=np.int64)
    for t in range(1, n):
        cand = dp[:, None] + trans
        back[t] = cand.argmax(0); dp = cand.max(0) + score[t]
    path = [int(dp.argmax())]
    for t in range(n - 1, 0, -1):
        path.append(int(back[t, path[-1]]))
    return path[::-1]


def decode(lp, allowed=None, trans=None, lam=0.0):
    s = lp + (np.where(allowed, 0.0, -1e9) if allowed is not None else 0.0)
    if lam and trans is not None and len(s) > 1:
        return viterbi(s, lam * trans)
    return s.argmax(1).tolist()


# ----------------------------------------------------------------------------- inference + metrics
@torch.no_grad()
def predict(model, enc, keys, ctx="doc", W=512, device="cuda", bs=16):
    """{key: (n_sentences, NL) mean log-probs over all windows containing each sentence}."""
    model.eval()
    ex = sorted(make_examples(enc, keys, ctx, W), key=lambda x: len(x["ids"]))
    acc = {k: np.zeros((len(enc[k]["sents"]), NL)) for k in keys}
    cnt = {k: np.zeros(len(enc[k]["sents"])) for k in keys}
    for b in range(0, len(ex), bs):
        chunk = ex[b:b + bs]
        ids, seg, off, _, pos = collate(chunk, device)
        with _autocast(device):
            lp = model(ids, seg, off, pos).float().log_softmax(-1).cpu().numpy()
        j = 0
        for x in chunk:
            for i in x["sidx"]:
                acc[x["key"]][i] += lp[j]; cnt[x["key"]][i] += 1; j += 1
    return {k: acc[k] / cnt[k][:, None] for k in keys}


def scores(gold, pred):
    from sklearn.metrics import f1_score, precision_score, recall_score
    g, p = np.asarray(gold), np.asarray(pred)
    return {"macro_f1": f1_score(g, p, average="macro", zero_division=0),
            "macro_p": precision_score(g, p, average="macro", zero_division=0),
            "macro_r": recall_score(g, p, average="macro", zero_division=0),
            "acc": float((g == p).mean())}


def evaluate(LP, enc, keys, allowed=None, trans=None, lam=0.0):
    """Macro-F1 (sklearn: labels = gold U pred) for all dev sentences and per domain."""
    G, P = {"all": [], "CL": [], "IT": []}, {"all": [], "CL": [], "IT": []}
    for k in keys:
        dom = enc[k]["dom"]
        pr = decode(LP[k], None if allowed is None else allowed[dom],
                    None if trans is None else trans[dom], lam)
        for name in ("all", dom):
            G[name] += enc[k]["labs"]; P[name] += pr
    return {n: scores(G[n], P[n]) for n in G if G[n]}


# ----------------------------------------------------------------------------- training
def train_rr(model, train_ex, eval_fn, epochs=8, lr=5e-4, head_lr=1e-3, bs=8, class_weight=None,
             device="cuda", seed=0, warmup=0.06, log=print):
    """AdamW, warm-up + linear decay, fp16. Length-bucketed batches (little padding). After every
    epoch eval_fn(model) -> metrics dict; keeps the epoch with the best dev macro-F1 ('all')."""
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
    model.to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    fast = {id(p) for p in model.tagger.head.parameters()} | {id(p) for p in model.pos.parameters()}
    opt = torch.optim.AdamW([{"params": [p for p in params if id(p) not in fast], "lr": lr},
                             {"params": [p for p in params if id(p) in fast], "lr": head_lr}],
                            weight_decay=0.01)
    steps = epochs * math.ceil(len(train_ex) / bs); warm = max(1, int(warmup * steps))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min((s + 1) / warm, max(0.0, (steps - s) / max(1, steps - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=str(device).startswith("cuda"))
    w = None if class_weight is None else class_weight.to(device)
    best, best_state, hist = -1.0, None, []
    log(f"trainable {sum(p.numel() for p in params)/1e6:.3f}M | {len(train_ex)} windows | {steps} steps")
    for ep in range(epochs):
        model.train(); t0 = time.time(); tot = 0.0
        order = sorted(range(len(train_ex)), key=lambda i: len(train_ex[i]["ids"]) + random.random() * 16)
        batches = [order[i:i + bs] for i in range(0, len(order), bs)]
        random.shuffle(batches)
        for bidx in batches:
            ids, seg, off, labs, pos = collate([train_ex[i] for i in bidx], device)
            with _autocast(device):
                logits = model(ids, seg, off, pos)
            loss = F.cross_entropy(logits.float(), labs, weight=w, ignore_index=-100)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update(); sched.step(); tot += loss.item()
        r = eval_fn(model)
        hist.append({"epoch": ep + 1, "loss": tot / len(batches), **{k: v["macro_f1"] for k, v in r.items()}})
        log(f"epoch {ep+1}: loss {hist[-1]['loss']:.4f} | dev macro-F1 all {r['all']['macro_f1']:.4f} "
            f"(CL {r['CL']['macro_f1']:.4f} IT {r['IT']['macro_f1']:.4f}) acc {r['all']['acc']:.3f} | {time.time()-t0:.0f}s")
        if r["all"]["macro_f1"] > best:
            best = r["all"]["macro_f1"]
            best_state = {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}
    model.load_state_dict({k: v.to(device) for k, v in best_state.items()}, strict=False)
    return hist, best

Overwriting /kaggle/working/src/p3_rr.py


In [8]:
# ============================================================================================
# P3-2 — Phase-3 setup + baselines: sentence-only vs document context (vs context + position)
#   Self-contained: re-creates WORK, the tokenizer and the data if P3-0 is not in memory.
#   M1 + LoRA (r=8, q,v), bidirectional, CL+IT joint.
# ============================================================================================
import sys, os, json, time, shutil, importlib, glob
import numpy as np, torch

WORK = "/kaggle/working"
sys.path.insert(0, f"{WORK}/src")
import p3_rr as rr; importlib.reload(rr)

# ---- tokenizer + data (re-load if P3-0 was run in another kernel) --------------------------------
if "tk" not in globals():
    from tokenizers import Tokenizer
    tk = Tokenizer.from_file(f"{WORK}/tokenizer/hf_a1_bpe_v2_32k.json")
if not all(f"{d}_{s}" in globals().get("D", {}) for d in ("CL", "IT") for s in ("train", "dev", "test")):
    def load_docs(path):
        """List of docs, or dict id -> doc / id -> [sentences] (test files): always {id, text, labels?}."""
        obj = json.load(open(path))
        if isinstance(obj, dict):
            obj = [{"id": k, **v} if isinstance(v, dict) else {"id": k, "text": v} for k, v in obj.items()]
        return obj
    D = {os.path.basename(f)[:-5]: load_docs(f)
         for f in glob.glob(f"{WORK}/data_p3/Phase_3_RR/*/*.json") if not f.endswith("label_vocab.json")}
print({k: len(v) for k, v in sorted(D.items())}, "docs")          # expect 40 / 5 / 5 per domain

DEV = "cuda:0"; torch.cuda.set_device(0)
CK = {"m1": f"{WORK}/checkpoints/m1_dense_100m/best.pt", "m2": f"{WORK}/checkpoints/m2_mla_moe/best.pt",
      "m3": f"{WORK}/checkpoints/m3_encdec/best.pt",     "m4": f"{WORK}/checkpoints/m4_copy/best.pt"}
os.makedirs(f"{WORK}/p3_ckpt", exist_ok=True); os.makedirs(f"{WORK}/stats", exist_ok=True)
print(f"free disk: {shutil.disk_usage(WORK).free/1e9:.1f} GB")

# ---- 1. encode all splits once (keys are 'CL::id' / 'IT::id') ---------------------------------
ENC, KEYS = {}, {"train": [], "dev": [], "test": []}
for dom in ["CL", "IT"]:
    for split in ["train", "dev", "test"]:
        e = rr.encode_docs(D[f"{dom}_{split}"], tk, dom)
        ENC.update(e); KEYS[split] += list(e)
print({s: len(k) for s, k in KEYS.items()}, "docs")

# ---- 2. domain priors (label masks + transitions) and class weights from TRAIN only ------------
ALLOWED, TRANS = rr.domain_stats(ENC, KEYS["train"])
for dom in ALLOWED:
    print(f"{dom}: never predicts {[rr.LABELS[i] for i in range(rr.NL) if not ALLOWED[dom][i]]}")
cnt = np.bincount([l for k in KEYS["train"] for l in ENC[k]["labs"]], minlength=rr.NL).astype(float)
CW = torch.tensor(np.where(cnt > 0, np.sqrt(cnt.max() / np.maximum(cnt, 1)), 1.0), dtype=torch.float).clamp(max=10.0)

# ---- 3. one experiment = train, pick best epoch on dev, evaluate with Viterbi, save --------------
RES_PATH = f"{WORK}/stats/p3_results.json"
RES = json.load(open(RES_PATH)) if os.path.exists(RES_PATH) else {}

def run(name, ckpt="m1", method="lora", ctx="doc", W=512, use_pos=True, bidir=True,
        epochs=8, lr=5e-4, bs=None, seed=0, **kw):
    t0 = time.time(); bs = bs or (32 if ctx == "none" else 8)
    model = rr.build_rr(CK[ckpt], method, bidirectional=bidir, use_pos=use_pos, **kw)
    tr_ex = rr.make_examples(ENC, KEYS["train"], ctx, W)
    ev = lambda m: rr.evaluate(rr.predict(m, ENC, KEYS["dev"], ctx, W, DEV), ENC, KEYS["dev"], ALLOWED)
    print(f"\n=== {name}: {ckpt} | {method} | ctx={ctx} W={W} pos={use_pos} bidir={bidir} ===")
    hist, best = rr.train_rr(model, tr_ex, ev, epochs=epochs, lr=lr, bs=bs, class_weight=CW,
                             device=DEV, seed=seed)
    LP = rr.predict(model, ENC, KEYS["dev"], ctx, W, DEV)                     # best epoch's dev log-probs
    out = {"cfg": dict(ckpt=ckpt, method=method, ctx=ctx, W=W, use_pos=use_pos, bidir=bidir,
                       epochs=epochs, lr=lr, seed=seed, **kw),
           "trainable_M": round(sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6, 3),
           "hist": hist, "min": round((time.time() - t0) / 60, 1)}
    for lam in (0.0, 0.5, 1.0):                                               # transition Viterbi weight
        r = rr.evaluate(LP, ENC, KEYS["dev"], ALLOWED, TRANS, lam)
        out[f"lam{lam}"] = {k: {m: round(v, 4) for m, v in d.items()} for k, d in r.items()}
    RES[name] = out; json.dump(RES, open(RES_PATH, "w"), indent=1)
    rr.save_trainable(model, f"{WORK}/p3_ckpt/{name}.pt")
    print("   " + " | ".join(f"λ={lam}: all {out[f'lam{lam}']['all']['macro_f1']:.4f} "
                             f"(CL {out[f'lam{lam}']['CL']['macro_f1']:.4f} IT {out[f'lam{lam}']['IT']['macro_f1']:.4f})"
                             for lam in (0.0, 0.5, 1.0)) + f" | {out['min']} min")
    del model; torch.cuda.empty_cache()
    return LP

# ---- 4. the three baselines ------------------------------------------------------------------
LPS = {}
LPS["m1_lora_sent"]    = run("m1_lora_sent",    ctx="none", use_pos=False)   # sentence only
LPS["m1_lora_doc"]     = run("m1_lora_doc",     ctx="doc",  use_pos=False)   # + document context
LPS["m1_lora_doc_pos"] = run("m1_lora_doc_pos", ctx="doc",  use_pos=True)    # + relative position

# ---- 5. summary --------------------------------------------------------------------------------
print(f"\n{'run':<18}{'λ':>5}{'all':>8}{'CL':>8}{'IT':>8}{'acc':>7}")
for n in LPS:
    for lam in (0.0, 0.5, 1.0):
        r = RES[n][f"lam{lam}"]
        print(f"{n:<18}{lam:>5}{r['all']['macro_f1']:>8.4f}{r['CL']['macro_f1']:>8.4f}"
              f"{r['IT']['macro_f1']:>8.4f}{r['all']['acc']:>7.3f}")

{'CL_dev': 5, 'CL_test': 5, 'CL_train': 40, 'IT_dev': 5, 'IT_test': 5, 'IT_train': 40} docs
free disk: 3.5 GB
{'train': 80, 'dev': 10, 'test': 10} docs
CL: never predicts ['PrecedentOverruled']
IT: never predicts ['Dissent']

=== m1_lora_sent: m1 | lora | ctx=none W=512 pos=False bidir=True ===
trainable 0.263M | 16845 windows | 4216 steps


/kaggle/working/src/p3_rr.py:331: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step(); tot += loss.item()


epoch 1: loss 1.8000 | dev macro-F1 all 0.3460 (CL 0.2466 IT 0.4653) acc 0.399 | 55s
epoch 2: loss 1.3578 | dev macro-F1 all 0.3471 (CL 0.2676 IT 0.4243) acc 0.397 | 54s
epoch 3: loss 1.1872 | dev macro-F1 all 0.3566 (CL 0.2532 IT 0.4554) acc 0.418 | 55s
epoch 4: loss 1.0418 | dev macro-F1 all 0.3767 (CL 0.2905 IT 0.4664) acc 0.424 | 55s
epoch 5: loss 0.9072 | dev macro-F1 all 0.3707 (CL 0.2961 IT 0.4513) acc 0.423 | 55s
epoch 6: loss 0.7922 | dev macro-F1 all 0.3589 (CL 0.2624 IT 0.4461) acc 0.411 | 55s
epoch 7: loss 0.6955 | dev macro-F1 all 0.3701 (CL 0.3001 IT 0.4460) acc 0.416 | 56s
epoch 8: loss 0.6348 | dev macro-F1 all 0.3622 (CL 0.2689 IT 0.4419) acc 0.414 | 55s
   λ=0.0: all 0.3767 (CL 0.2905 IT 0.4664) | λ=0.5: all 0.4388 (CL 0.3116 IT 0.4912) | λ=1.0: all 0.4461 (CL 0.3521 IT 0.4913) | 7.5 min

=== m1_lora_doc: m1 | lora | ctx=doc W=512 pos=False bidir=True ===
trainable 0.263M | 3129 windows | 3136 steps


/kaggle/working/src/p3_rr.py:331: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step(); tot += loss.item()


epoch 1: loss 1.5798 | dev macro-F1 all 0.4448 (CL 0.3675 IT 0.5097) acc 0.513 | 95s
epoch 2: loss 0.8934 | dev macro-F1 all 0.5105 (CL 0.4333 IT 0.5173) acc 0.540 | 95s
epoch 3: loss 0.6299 | dev macro-F1 all 0.5258 (CL 0.4801 IT 0.5310) acc 0.556 | 95s
epoch 4: loss 0.4498 | dev macro-F1 all 0.5292 (CL 0.4700 IT 0.5483) acc 0.553 | 95s
epoch 5: loss 0.3300 | dev macro-F1 all 0.5318 (CL 0.4574 IT 0.5502) acc 0.563 | 94s
epoch 6: loss 0.2423 | dev macro-F1 all 0.5074 (CL 0.4399 IT 0.5297) acc 0.545 | 95s
epoch 7: loss 0.1906 | dev macro-F1 all 0.5197 (CL 0.4493 IT 0.5400) acc 0.552 | 95s
epoch 8: loss 0.1572 | dev macro-F1 all 0.5198 (CL 0.4535 IT 0.5402) acc 0.551 | 95s
   λ=0.0: all 0.5318 (CL 0.4574 IT 0.5502) | λ=0.5: all 0.5326 (CL 0.4377 IT 0.5355) | λ=1.0: all 0.5148 (CL 0.4063 IT 0.5194) | 12.7 min

=== m1_lora_doc_pos: m1 | lora | ctx=doc W=512 pos=True bidir=True ===
trainable 0.263M | 3129 windows | 3136 steps


/kaggle/working/src/p3_rr.py:331: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step(); tot += loss.item()


epoch 1: loss 1.5480 | dev macro-F1 all 0.4667 (CL 0.4027 IT 0.5234) acc 0.530 | 95s
epoch 2: loss 0.8412 | dev macro-F1 all 0.5232 (CL 0.4608 IT 0.5156) acc 0.565 | 95s
epoch 3: loss 0.5879 | dev macro-F1 all 0.5444 (CL 0.4873 IT 0.5627) acc 0.565 | 95s
epoch 4: loss 0.4222 | dev macro-F1 all 0.5273 (CL 0.4851 IT 0.5414) acc 0.561 | 95s
epoch 5: loss 0.3080 | dev macro-F1 all 0.5403 (CL 0.4674 IT 0.5592) acc 0.572 | 95s
epoch 6: loss 0.2233 | dev macro-F1 all 0.5096 (CL 0.3988 IT 0.5355) acc 0.545 | 95s
epoch 7: loss 0.1719 | dev macro-F1 all 0.5145 (CL 0.3973 IT 0.5495) acc 0.560 | 95s
epoch 8: loss 0.1416 | dev macro-F1 all 0.5111 (CL 0.3797 IT 0.5489) acc 0.556 | 95s
   λ=0.0: all 0.5444 (CL 0.4873 IT 0.5627) | λ=0.5: all 0.5520 (CL 0.4438 IT 0.5715) | λ=1.0: all 0.5351 (CL 0.4505 IT 0.5286) | 12.7 min

run                   λ     all      CL      IT    acc
m1_lora_sent        0.0  0.3767  0.2905  0.4664  0.424
m1_lora_sent        0.5  0.4388  0.3116  0.4912  0.496
m1_lora_sent    

## P3-3 · Technique × architecture comparison

**P3-2 result (M1 + LoRA, dev macro-F1):** sentence-only 0.377 → + document context **0.532** (+0.155; CL +0.17) → + relative position **0.544**. Transition Viterbi helps the sentence-only model (+0.07) but adds nothing on top of document context (the windowed model has already learned the role transitions). Dev peaks at epochs 3–5, so from now on we use 6 epochs.

**Fixed recipe:** document-context windows (W = 512, 50% overlap), position embedding, class-weighted CE, domain label masks, joint CL+IT training, 6 epochs, best epoch by dev macro-F1.

**Runs** (techniques from §9.2.1; the learning rate is tuned per technique, as the spec allows):

| run | model | technique | lr |
|---|---|---|---|
| m1_rslora | M1 | rsLoRA (r = 16, α/√r) | 2e-4 |
| m1_lorma_plus | M1 | **LoRMA+** | 5e-4 |
| m1_ia3 | M1 | IA³ | 3e-3 |
| m1_prompt | M1 | prompt tuning (20 vectors) | 1e-2 |
| m2_lorma_plus | **M2** | **LoRMA+** | 5e-4 |
| m3_lora | **M3** (encoder) | LoRA | 5e-4 |
| m4_lora | **M4** | LoRA | 5e-4 |
| m1_full | M1 | full fine-tuning (reference, not a listed PEFT method) | 5e-5 |

KD comes next, in P3-4.

In [ ]:
# ============================================================================================
# P3-3 — Technique x architecture comparison with the fixed recipe (context + position, 6 epochs)
#   Uses run(), ENC, KEYS, RES from P3-2 (same kernel). Each run is saved to p3_results.json as it
#   finishes, so an interruption loses at most the current run (finished ones are skipped on re-run).
# ============================================================================================
import shutil
shutil.rmtree(f"{WORK}/p2_ckpt", ignore_errors=True)               # Phase-2 ckpts are safe on HF
print(f"free disk: {shutil.disk_usage(WORK).free/1e9:.1f} GB")

PLAN = [   # name,            model, technique,    lr,    extra kwargs
    ("m1_rslora",       "m1", "rslora",     2e-4, {"rs_r": 16}),
    ("m1_lorma_plus",   "m1", "lorma_plus", 5e-4, {}),
    ("m1_ia3",          "m1", "ia3",        3e-3, {}),
    ("m1_prompt",       "m1", "prompt",     1e-2, {}),
    ("m2_lorma_plus",   "m2", "lorma_plus", 5e-4, {}),
    ("m3_lora",         "m3", "lora",       5e-4, {}),
    ("m4_lora",         "m4", "lora",       5e-4, {}),
    ("m1_full",         "m1", "full",       5e-5, {}),
]
for name, ck, method, lr, kw in PLAN:
    if name in RES:                                                 # resume-safe
        print(f"skip {name} (done)"); continue
    try:
        LPS[name] = run(name, ckpt=ck, method=method, ctx="doc", W=512, use_pos=True,
                        epochs=6, lr=lr, **kw)
    except Exception as e:                                          # one failing method must not stop the rest
        print(f"!!! {name} failed: {type(e).__name__}: {e}")
        torch.cuda.empty_cache()

# ---- summary: every run so far, sorted by dev macro-F1 (λ=0), with λ=0.5 alongside -------------
print(f"\n{'run':<18}{'trainable':>11}{'all λ0':>8}{'all λ.5':>8}{'CL':>8}{'IT':>8}{'P':>7}{'R':>7}{'min':>6}")
for n, r in sorted(RES.items(), key=lambda kv: -kv[1]["lam0.0"]["all"]["macro_f1"]):
    a, b = r["lam0.0"], r["lam0.5"]
    print(f"{n:<18}{r['trainable_M']:>10}M{a['all']['macro_f1']:>8.4f}{b['all']['macro_f1']:>8.4f}"
          f"{a['CL']['macro_f1']:>8.4f}{a['IT']['macro_f1']:>8.4f}{a['all']['macro_p']:>7.3f}"
          f"{a['all']['macro_r']:>7.3f}{r['min']:>6}")

free disk: 10.1 GB

=== m1_rslora: m1 | rslora | ctx=doc W=512 pos=True bidir=True ===
trainable 0.509M | 3129 windows | 2352 steps


/kaggle/working/src/p3_rr.py:331: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step(); tot += loss.item()


epoch 1: loss 1.4821 | dev macro-F1 all 0.4699 (CL 0.3874 IT 0.5174) acc 0.538 | 95s
epoch 2: loss 0.7868 | dev macro-F1 all 0.5013 (CL 0.4691 IT 0.5217) acc 0.597 | 95s
epoch 3: loss 0.5194 | dev macro-F1 all 0.5550 (CL 0.5192 IT 0.5578) acc 0.583 | 95s
epoch 4: loss 0.3579 | dev macro-F1 all 0.5261 (CL 0.4258 IT 0.5399) acc 0.570 | 95s
epoch 5: loss 0.2567 | dev macro-F1 all 0.5337 (CL 0.4313 IT 0.5557) acc 0.578 | 95s
epoch 6: loss 0.1936 | dev macro-F1 all 0.5305 (CL 0.4317 IT 0.5520) acc 0.577 | 95s
   λ=0.0: all 0.5550 (CL 0.5192 IT 0.5578) | λ=0.5: all 0.5542 (CL 0.4936 IT 0.5646) | λ=1.0: all 0.5365 (CL 0.5059 IT 0.5319) | 9.6 min

=== m1_lorma_plus: m1 | lorma_plus | ctx=doc W=512 pos=True bidir=True ===
trainable 0.214M | 3129 windows | 2352 steps


/kaggle/working/src/p3_rr.py:331: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step(); tot += loss.item()


epoch 1: loss 1.4621 | dev macro-F1 all 0.4709 (CL 0.3782 IT 0.5365) acc 0.541 | 92s
epoch 2: loss 0.8027 | dev macro-F1 all 0.5357 (CL 0.4941 IT 0.5179) acc 0.586 | 92s
epoch 3: loss 0.5747 | dev macro-F1 all 0.5503 (CL 0.4934 IT 0.5571) acc 0.588 | 92s
epoch 4: loss 0.4392 | dev macro-F1 all 0.5259 (CL 0.4198 IT 0.5448) acc 0.579 | 92s
epoch 5: loss 0.3481 | dev macro-F1 all 0.5167 (CL 0.4247 IT 0.5176) acc 0.576 | 92s
epoch 6: loss 0.2881 | dev macro-F1 all 0.5200 (CL 0.4257 IT 0.5307) acc 0.583 | 92s
   λ=0.0: all 0.5503 (CL 0.4934 IT 0.5571) | λ=0.5: all 0.5472 (CL 0.4274 IT 0.5758) | λ=1.0: all 0.5023 (CL 0.4569 IT 0.5175) | 9.3 min

=== m1_ia3: m1 | ia3 | ctx=doc W=512 pos=True bidir=True ===
trainable 0.048M | 3129 windows | 2352 steps


/kaggle/working/src/p3_rr.py:331: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step(); tot += loss.item()


epoch 1: loss 1.5962 | dev macro-F1 all 0.4532 (CL 0.3628 IT 0.4998) acc 0.516 | 99s
epoch 2: loss 0.9264 | dev macro-F1 all 0.4615 (CL 0.4100 IT 0.4885) acc 0.553 | 99s
epoch 3: loss 0.7192 | dev macro-F1 all 0.4996 (CL 0.4365 IT 0.5471) acc 0.583 | 99s
epoch 4: loss 0.6089 | dev macro-F1 all 0.4925 (CL 0.4141 IT 0.5254) acc 0.579 | 99s
epoch 5: loss 0.5351 | dev macro-F1 all 0.4874 (CL 0.4025 IT 0.5402) acc 0.569 | 99s
epoch 6: loss 0.4867 | dev macro-F1 all 0.4942 (CL 0.4233 IT 0.5440) acc 0.584 | 99s
   λ=0.0: all 0.4996 (CL 0.4365 IT 0.5471) | λ=0.5: all 0.4960 (CL 0.3880 IT 0.5364) | λ=1.0: all 0.4828 (CL 0.4155 IT 0.5200) | 10.0 min

=== m1_prompt: m1 | prompt | ctx=doc W=512 pos=True bidir=True ===
trainable 0.033M | 3129 windows | 2352 steps


/kaggle/working/src/p3_rr.py:331: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step(); tot += loss.item()


epoch 1: loss 1.6740 | dev macro-F1 all 0.3895 (CL 0.2953 IT 0.4423) acc 0.465 | 94s
epoch 2: loss 1.1345 | dev macro-F1 all 0.4056 (CL 0.3233 IT 0.4579) acc 0.504 | 94s
epoch 3: loss 0.9861 | dev macro-F1 all 0.4217 (CL 0.3331 IT 0.4839) acc 0.517 | 94s
epoch 4: loss 0.9032 | dev macro-F1 all 0.4424 (CL 0.3466 IT 0.4866) acc 0.537 | 94s
epoch 5: loss 0.8508 | dev macro-F1 all 0.4414 (CL 0.3583 IT 0.4941) acc 0.526 | 94s
epoch 6: loss 0.8048 | dev macro-F1 all 0.4364 (CL 0.3433 IT 0.5077) acc 0.527 | 94s
   λ=0.0: all 0.4424 (CL 0.3466 IT 0.4866) | λ=0.5: all 0.4446 (CL 0.3846 IT 0.4601) | λ=1.0: all 0.4345 (CL 0.3820 IT 0.4380) | 9.5 min

=== m2_lorma_plus: m2 | lorma_plus | ctx=doc W=512 pos=True bidir=True ===
trainable 0.313M | 3129 windows | 2352 steps
epoch 1: loss 1.4290 | dev macro-F1 all 0.4541 (CL 0.3654 IT 0.5146) acc 0.514 | 135s
epoch 2: loss 0.7545 | dev macro-F1 all 0.4683 (CL 0.3901 IT 0.5146) acc 0.548 | 135s
epoch 3: loss 0.5307 | dev macro-F1 all 0.4852 (CL 0.4261 IT

In [10]:
# ============================================================================================
# P3-W — Progress + results table (reads logs and result files; never touches training)
# ============================================================================================
import json, glob, os, subprocess
WORK = "/kaggle/working"
print(subprocess.run("echo 'p3_run processes:' $(pgrep -fc '[p]3_run.py'); "
                     "nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout)
for g in (0, 1):
    lines = open(f"{WORK}/stats/p3_gpu{g}.log").read().splitlines() if os.path.exists(f"{WORK}/stats/p3_gpu{g}.log") else []
    print(f"--- GPU {g} (last 4 lines)"); [print("  ", l[:170]) for l in lines[-4:]]

R = json.load(open(f"{WORK}/stats/p3_results.json")) if os.path.exists(f"{WORK}/stats/p3_results.json") else {}
for f in glob.glob(f"{WORK}/stats/p3_runs/*.json"):
    R[os.path.basename(f)[:-5]] = json.load(open(f))
print(f"\n{'run':<18}{'trainable':>11}{'all λ0':>8}{'all λ.5':>8}{'CL':>8}{'IT':>8}{'P':>7}{'R':>7}{'min':>6}")
for n, r in sorted(R.items(), key=lambda kv: -kv[1]["lam0.0"]["all"]["macro_f1"]):
    a, b = r["lam0.0"], r["lam0.5"]
    print(f"{n:<18}{r.get('trainable_M','?'):>10}M{a['all']['macro_f1']:>8.4f}{b['all']['macro_f1']:>8.4f}"
          f"{a['CL']['macro_f1']:>8.4f}{a['IT']['macro_f1']:>8.4f}{a['all']['macro_p']:>7.3f}"
          f"{a['all']['macro_r']:>7.3f}{r['min']:>6}")

p3_run processes: 0
0, 0 %, 185 MiB
1, 0 %, 3 MiB

--- GPU 0 (last 4 lines)
--- GPU 1 (last 4 lines)

run                 trainable  all λ0 all λ.5      CL      IT      P      R   min
m1_rslora              0.509M  0.5550  0.5542  0.5192  0.5578  0.573  0.551   9.6
m1_lorma_plus          0.214M  0.5503  0.5472  0.4934  0.5571  0.553  0.557   9.3
m1_full              100.112M  0.5472  0.5297  0.4628  0.5780  0.555  0.555  12.2
m1_lora_doc_pos        0.263M  0.5444  0.5520  0.4873  0.5627  0.553  0.548  12.7
m1_lora_doc            0.263M  0.5318  0.5326  0.4574  0.5502  0.548  0.536  12.7
m2_lorma_plus          0.313M  0.5268  0.5315  0.4408  0.5354  0.533  0.543  13.7
m4_lora                0.263M  0.5231  0.5503  0.4627  0.5535  0.531  0.538   9.6
m1_ia3                 0.048M  0.4996  0.4960  0.4365  0.5471  0.501  0.505  10.0
m1_prompt              0.033M  0.4424  0.4446  0.3466  0.4866  0.461  0.447   9.5
m3_lora                0.141M  0.3797  0.3796  0.3644  0.4365  0.384  0.398   

## P3-R · Background runner (`p3_run.py`)

Long cells were being cut off when the browser connection dropped. Experiments now run as **background processes**, one per GPU, which survive reloads and kernel restarts. Each finished run writes its metrics (`stats/p3_runs/<name>.json`), its trained parameters (`p3_ckpt/`) and its dev and test log-probs (`p3_lp/`, for ensembling, KD and submissions). Finished runs are skipped on re-launch.

In [13]:
%%writefile /kaggle/working/src/p3_run.py
"""
p3_run.py -- run a list of Phase-3 experiments as a BACKGROUND process (survives browser reloads,
cell interrupts and kernel restarts). Launch one process per GPU:

    CUDA_VISIBLE_DEVICES=1 PYTHONUNBUFFERED=1 setsid nohup python p3_run.py --plan plan.json > log 2>&1 &

plan.json = [{"name": "m1_rslora", "ckpt": "m1", "method": "rslora", "lr": 2e-4, "rs_r": 16}, ...]
Optional per-run keys: ctx ("doc"/"none"), W, use_pos, bidir, epochs, bs, seed + method kwargs.

Per finished run it writes
    stats/p3_runs/<name>.json   config, trainable params, per-epoch history, dev metrics at lambda 0/0.5/1
    p3_ckpt/<name>.pt           trained parameters only (PEFT: a few MB)
    p3_lp/<name>.pkl            dev + test log-probs per document (for ensembling / KD / submissions)
Runs whose result file already exists (or that are in the old stats/p3_results.json) are skipped.

Two extra job types:
  {"type": "lp", "name": X, "train": true}   re-load finished run X and (re)compute its log-probs
                                             (train split only if asked: needed for KD teachers)
  KD run: normal run + "teachers": [names], "student_layers": [0,2,4,6,8,10], "kd_T", "kd_lam"
          (teacher = mean of the teachers' TRAIN log-probs; student = M1 with only those layers)
"""
import argparse
import glob
import json
import os
import pickle
import sys
import time
import traceback

import numpy as np
import torch

WORK = "/kaggle/working"
sys.path.insert(0, f"{WORK}/src")
import p3_rr as rr                                                    # noqa: E402
import p2_peft as pp                                                  # noqa: E402
from tokenizers import Tokenizer                                     # noqa: E402

ap = argparse.ArgumentParser()
ap.add_argument("--plan", required=True)
PLAN = json.load(open(ap.parse_args().plan))

DEV = "cuda:0"                                                       # = the GPU given by CUDA_VISIBLE_DEVICES
CK = {"m1": f"{WORK}/checkpoints/m1_dense_100m/best.pt", "m2": f"{WORK}/checkpoints/m2_mla_moe/best.pt",
      "m3": f"{WORK}/checkpoints/m3_encdec/best.pt", "m4": f"{WORK}/checkpoints/m4_copy/best.pt"}
OUT, LPD, CKD = f"{WORK}/stats/p3_runs", f"{WORK}/p3_lp", f"{WORK}/p3_ckpt"
for d in (OUT, LPD, CKD):
    os.makedirs(d, exist_ok=True)


def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)


# ---- data (same encoding, priors and class weights as the notebook) ----------------------------
def load_docs(path):
    obj = json.load(open(path))
    if isinstance(obj, dict):
        obj = [{"id": k, **v} if isinstance(v, dict) else {"id": k, "text": v} for k, v in obj.items()]
    return obj


tk = Tokenizer.from_file(f"{WORK}/tokenizer/hf_a1_bpe_v2_32k.json")
D = {os.path.basename(f)[:-5]: load_docs(f)
     for f in glob.glob(f"{WORK}/data_p3/Phase_3_RR/*/*.json") if not f.endswith("label_vocab.json")}
ENC, KEYS = {}, {"train": [], "dev": [], "test": []}
for dom in ["CL", "IT"]:
    for split in ["train", "dev", "test"]:
        e = rr.encode_docs(D[f"{dom}_{split}"], tk, dom)
        ENC.update(e); KEYS[split] += list(e)
ALLOWED, TRANS = rr.domain_stats(ENC, KEYS["train"])
cnt = np.bincount([l for k in KEYS["train"] for l in ENC[k]["labs"]], minlength=rr.NL).astype(float)
CW = torch.tensor(np.where(cnt > 0, np.sqrt(cnt.max() / np.maximum(cnt, 1)), 1.0), dtype=torch.float).clamp(max=10.0)
OLD = json.load(open(f"{WORK}/stats/p3_results.json")) if os.path.exists(f"{WORK}/stats/p3_results.json") else {}
log(f"data ready: {[len(KEYS[s]) for s in KEYS]} docs | plan: {[p['name'] for p in PLAN]}")


def run(name, ckpt="m1", method="lora", ctx="doc", W=512, use_pos=True, bidir=True,
        epochs=6, lr=5e-4, bs=None, seed=0, teachers=None, student_layers=None, kd_T=2.0, kd_lam=0.5, **kw):
    t0 = time.time(); bs = bs or (32 if ctx == "none" else 8)
    model = rr.build_rr(CK[ckpt], method, bidirectional=bidir, use_pos=use_pos, **kw)
    if student_layers:                                               # smaller student: keep a subset of layers
        pp.shrink_dense(model.tagger, tuple(student_layers))
    tr_ex = rr.make_examples(ENC, KEYS["train"], ctx, W)
    if teachers:                                                     # soft targets = mean teacher train log-probs
        T = [pickle.load(open(f"{LPD}/{t}.pkl", "rb"))["train"] for t in teachers]
        soft = {k: np.mean([t[k] for t in T], 0) for k in KEYS["train"]}
        for x in tr_ex:
            x["soft"] = [soft[x["key"]][i] for i in x["sidx"]]
    ev = lambda m: rr.evaluate(rr.predict(m, ENC, KEYS["dev"], ctx, W, DEV), ENC, KEYS["dev"], ALLOWED)
    log(f"=== {name}: {ckpt} | {method} | ctx={ctx} W={W} pos={use_pos} bidir={bidir} lr={lr} ===")
    hist, best = rr.train_rr(model, tr_ex, ev, epochs=epochs, lr=lr, bs=bs, class_weight=CW,
                             device=DEV, seed=seed, log=log, kd_T=kd_T, kd_lam=kd_lam)
    LPd = rr.predict(model, ENC, KEYS["dev"], ctx, W, DEV)
    torch.cuda.synchronize(); ti = time.time()
    LPt = rr.predict(model, ENC, KEYS["test"], ctx, W, DEV)
    torch.cuda.synchronize(); infer_s = round(time.time() - ti, 2)
    out = {"cfg": dict(ckpt=ckpt, method=method, ctx=ctx, W=W, use_pos=use_pos, bidir=bidir,
                       epochs=epochs, lr=lr, seed=seed, teachers=teachers, student_layers=student_layers,
                       kd_T=kd_T, kd_lam=kd_lam, **kw),
           "trainable_M": round(sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6, 3),
           "total_M": round(sum(p.numel() for p in model.parameters()) / 1e6, 2), "infer_test_s": infer_s,
           "hist": hist, "min": round((time.time() - t0) / 60, 1)}
    for lam in (0.0, 0.5, 1.0):
        r = rr.evaluate(LPd, ENC, KEYS["dev"], ALLOWED, TRANS, lam)
        out[f"lam{lam}"] = {k: {m: round(v, 4) for m, v in d.items()} for k, d in r.items()}
    rr.save_trainable(model, f"{CKD}/{name}.pt")
    pickle.dump({"dev": LPd, "test": LPt}, open(f"{LPD}/{name}.pkl", "wb"))
    json.dump(out, open(f"{OUT}/{name}.json", "w"), indent=1)                # written LAST = "done" marker
    log(f"DONE {name}: dev macro λ0 {out['lam0.0']['all']['macro_f1']:.4f} "
        f"(CL {out['lam0.0']['CL']['macro_f1']:.4f} IT {out['lam0.0']['IT']['macro_f1']:.4f}) "
        f"| λ0.5 {out['lam0.5']['all']['macro_f1']:.4f} | {out['min']} min")
    del model; torch.cuda.empty_cache()


def lp_job(name, train=False):
    """Re-load finished run `name` (config from its result file) and save its log-probs."""
    R = json.load(open(f"{OUT}/{name}.json")) if os.path.exists(f"{OUT}/{name}.json") else OLD[name]
    c = {k: v for k, v in R["cfg"].items() if v is not None}
    ctx, W, ckpt, method = c.pop("ctx"), c.pop("W"), c.pop("ckpt"), c.pop("method")
    bidir, use_pos, layers = c.pop("bidir"), c.pop("use_pos"), c.pop("student_layers", None)
    for k in ("epochs", "lr", "seed", "bs", "teachers", "kd_T", "kd_lam"):
        c.pop(k, None)
    m = rr.build_rr(CK[ckpt], method, bidirectional=bidir, use_pos=use_pos, **c)
    if layers:
        pp.shrink_dense(m.tagger, tuple(layers))
    rr.load_trainable(m, f"{CKD}/{name}.pt"); m.to(DEV)
    path = f"{LPD}/{name}.pkl"
    out = pickle.load(open(path, "rb")) if os.path.exists(path) else {}
    for split in (["train"] if train else []) + ["dev", "test"]:
        if split not in out:
            ti = time.time(); out[split] = rr.predict(m, ENC, KEYS[split], ctx, W, DEV)
            log(f"   {name}: {split} log-probs in {time.time()-ti:.1f}s")
    pickle.dump(out, open(path, "wb"))
    chk = rr.evaluate(out["dev"], ENC, KEYS["dev"], ALLOWED)["all"]["macro_f1"]
    log(f"LP {name}: dev macro re-check {chk:.4f} (should equal its λ0 score)")
    del m; torch.cuda.empty_cache()


for cfg in PLAN:
    cfg = dict(cfg); name = cfg.pop("name")
    if cfg.pop("type", "run") == "lp":
        try:
            lp_job(name, cfg.get("train", False))
        except Exception:
            log(f"!!! LP {name} FAILED"); traceback.print_exc(); torch.cuda.empty_cache()
        continue
    if os.path.exists(f"{OUT}/{name}.json") or name in OLD:
        log(f"skip {name} (already done)"); continue
    try:
        run(name, **cfg)
    except Exception:
        log(f"!!! {name} FAILED"); traceback.print_exc(); torch.cuda.empty_cache()
log("PLAN DONE")

Overwriting /kaggle/working/src/p3_run.py


In [12]:
# Backup incl. Phase 3 (adds data_p3, p3_ckpt, p3_lp to the allow-list, then runs it)
import os, subprocess
from kaggle_secrets import UserSecretsClient
p = "/kaggle/working/src/backup_hf.py"
s = open(p).read()
if "p3_ckpt" not in s:
    s = s.replace('"submissions_p2/**",', '"submissions_p2/**", "data_p3/**", "p3_ckpt/**", "p3_lp/**", "submissions_p3/**",')
    open(p, "w").write(s)
env = {**os.environ, "HF_WRITE_TOKEN": UserSecretsClient().get_secret("HF_WRITE_TOKEN")}
print(subprocess.run(f"python {p}", shell=True, capture_output=True, text=True, env=env).stdout[-500:])

BACKUP DONE -> Utkarsh-1718-1/cs781-phase1-work in 36s



## P3-4 · Knowledge distillation

**Teacher:** mean posterior of the 4 best M1 models (rsLoRA, LoRMA+, full FT, LoRA), about 100M parameters each. **Student:** M1 with 6 of its 12 layers (layers 0, 2, 4, 6, 8, 10, as in DistilBERT's initialisation), fully fine-tuned with loss = 0.5·CE(gold) + 0.5·T²·KL(teacher_T ‖ student_T), T = 2. **Control:** the same student without KD. We compare macro-F1, parameter count and test inference time.

*Caveat:* the teachers' posteriors on the training sentences are more confident than on unseen text, because the teachers were trained on those sentences. The temperature T = 2 softens them.

In [14]:
%%writefile -a /kaggle/working/src/p3_rr.py


# ----------------------------------------------------------------------------- v2: training with optional KD
def train_rr(model, train_ex, eval_fn, epochs=8, lr=5e-4, head_lr=1e-3, bs=8, class_weight=None,
             device="cuda", seed=0, warmup=0.06, log=print, kd_T=2.0, kd_lam=0.5):
    """As v1, plus knowledge distillation: if the examples carry 'soft' (teacher log-probs per
    sentence), loss = (1 - kd_lam) * CE(gold) + kd_lam * T^2 * KL(teacher_T || student_T)."""
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
    model.to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    fast = {id(p) for p in model.tagger.head.parameters()} | {id(p) for p in model.pos.parameters()}
    opt = torch.optim.AdamW([{"params": [p for p in params if id(p) not in fast], "lr": lr},
                             {"params": [p for p in params if id(p) in fast], "lr": head_lr}],
                            weight_decay=0.01)
    steps = epochs * math.ceil(len(train_ex) / bs); warm = max(1, int(warmup * steps))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min((s + 1) / warm, max(0.0, (steps - s) / max(1, steps - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=str(device).startswith("cuda"))
    w = None if class_weight is None else class_weight.to(device)
    kd = "soft" in train_ex[0]
    best, best_state, hist = -1.0, None, []
    log(f"trainable {sum(p.numel() for p in params)/1e6:.3f}M | {len(train_ex)} windows | {steps} steps"
        + (f" | KD T={kd_T} lam={kd_lam}" if kd else ""))
    for ep in range(epochs):
        model.train(); t0 = time.time(); tot = 0.0
        order = sorted(range(len(train_ex)), key=lambda i: len(train_ex[i]["ids"]) + random.random() * 16)
        batches = [order[i:i + bs] for i in range(0, len(order), bs)]
        random.shuffle(batches)
        for bidx in batches:
            ids, seg, off, labs, pos = collate([train_ex[i] for i in bidx], device)
            with _autocast(device):
                logits = model(ids, seg, off, pos)
            loss = F.cross_entropy(logits.float(), labs, weight=w, ignore_index=-100)
            if kd:
                t = torch.tensor(np.concatenate([np.asarray(train_ex[i]["soft"]) for i in bidx]),
                                 dtype=torch.float, device=device)
                kl = F.kl_div(F.log_softmax(logits.float() / kd_T, -1), F.log_softmax(t / kd_T, -1),
                              log_target=True, reduction="batchmean") * kd_T * kd_T
                loss = (1 - kd_lam) * loss + kd_lam * kl
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update(); sched.step(); tot += loss.item()
        r = eval_fn(model)
        hist.append({"epoch": ep + 1, "loss": tot / len(batches), **{k: v["macro_f1"] for k, v in r.items()}})
        log(f"epoch {ep+1}: loss {hist[-1]['loss']:.4f} | dev macro-F1 all {r['all']['macro_f1']:.4f} "
            f"(CL {r['CL']['macro_f1']:.4f} IT {r['IT']['macro_f1']:.4f}) acc {r['all']['acc']:.3f} | {time.time()-t0:.0f}s")
        if r["all"]["macro_f1"] > best:
            best = r["all"]["macro_f1"]
            best_state = {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}
    model.load_state_dict({k: v.to(device) for k, v in best_state.items()}, strict=False)
    return hist, best

Appending to /kaggle/working/src/p3_rr.py


In [15]:
# ============================================================================================
# P3-4L — KD (GPU 0: teacher log-probs -> distilled 6-layer student) + no-KD control and the
#          log-probs of all other runs for ensembles/zips (GPU 1). Background; watch with P3-W.
# ============================================================================================
import json, subprocess
WORK = "/kaggle/working"
TEACHERS = ["m1_rslora", "m1_lorma_plus", "m1_full", "m1_lora_doc_pos"]
STUDENT = dict(ckpt="m1", method="full", lr=5e-5, student_layers=[0, 2, 4, 6, 8, 10])
PLANS = {
    0: [dict(type="lp", name=t, train=True) for t in TEACHERS] +                 # teacher posteriors (+dev/test)
       [dict(name="kd_student_m1x6", teachers=TEACHERS, kd_T=2.0, kd_lam=0.5, **STUDENT)],
    1: [dict(name="nokd_student_m1x6", **STUDENT)] +                             # same student, no teacher
       [dict(type="lp", name=n) for n in ["m2_lorma_plus", "m4_lora", "m3_lora", "m1_ia3",
                                          "m1_prompt", "m1_lora_doc", "m1_lora_sent"]],
}
if subprocess.run("pgrep -f '[p]3_run.py'", shell=True, capture_output=True, text=True).stdout.strip():
    print("p3_run.py is already running -> not launching again")
else:
    for g, plan in PLANS.items():
        pf = f"{WORK}/stats/p3_plan4_gpu{g}.json"; json.dump(plan, open(pf, "w"))
        subprocess.run(f"cd {WORK}/src && CUDA_VISIBLE_DEVICES={g} PYTHONUNBUFFERED=1 setsid nohup "
                       f"python p3_run.py --plan {pf} >> {WORK}/stats/p3_gpu{g}.log 2>&1 < /dev/null &", shell=True)
    print("launched P3-4 on both GPUs -> run P3-W to watch")

launched P3-4 on both GPUs -> run P3-W to watch


In [21]:
# ============================================================================================
# P3-W — Progress + results table (reads logs and result files; never touches training)
# ============================================================================================
import json, glob, os, subprocess
WORK = "/kaggle/working"

# ---- 1. are the background jobs alive, and are the GPUs busy? ---------------------------------
print(subprocess.run("echo 'p3_run processes:' $(pgrep -fc '[p]3_run.py'); "
                     "nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout)

# ---- 2. per GPU: last lines + every DONE / LP re-check / FAILED line ----------------------------
for g in (0, 1):
    p = f"{WORK}/stats/p3_gpu{g}.log"
    lines = open(p).read().splitlines() if os.path.exists(p) else []
    print(f"--- GPU {g}: key events")
    for l in lines:
        if any(k in l for k in ("DONE", "re-check", "FAILED", "skip", "Error")):
            print("   ", l[:170])
    print(f"--- GPU {g}: last 3 lines")
    for l in lines[-3:]:
        print("   ", l[:170])

# ---- 3. results table: old notebook runs + background runs, sorted by dev macro-F1 --------------
R = json.load(open(f"{WORK}/stats/p3_results.json")) if os.path.exists(f"{WORK}/stats/p3_results.json") else {}
for f in glob.glob(f"{WORK}/stats/p3_runs/*.json"):
    R[os.path.basename(f)[:-5]] = json.load(open(f))
print(f"\n{'run':<20}{'trainable':>11}{'total':>8}{'all λ0':>8}{'all λ.5':>8}{'CL':>8}{'IT':>8}"
      f"{'P':>7}{'R':>7}{'test s':>8}{'min':>6}")
for n, r in sorted(R.items(), key=lambda kv: -kv[1]["lam0.0"]["all"]["macro_f1"]):
    a, b = r["lam0.0"], r["lam0.5"]
    print(f"{n:<20}{r.get('trainable_M', '?'):>10}M{str(r.get('total_M', '')):>8}"
          f"{a['all']['macro_f1']:>8.4f}{b['all']['macro_f1']:>8.4f}{a['CL']['macro_f1']:>8.4f}"
          f"{a['IT']['macro_f1']:>8.4f}{a['all']['macro_p']:>7.3f}{a['all']['macro_r']:>7.3f}"
          f"{str(r.get('infer_test_s', '')):>8}{r['min']:>6}")

p3_run processes: 0
0, 0 %, 185 MiB
1, 0 %, 3 MiB

--- GPU 0: key events
    19:30:54 LP m1_rslora: dev macro re-check 0.5550 (should equal its λ0 score)
    19:31:39 LP m1_lorma_plus: dev macro re-check 0.5503 (should equal its λ0 score)
    19:32:21 LP m1_full: dev macro re-check 0.5468 (should equal its λ0 score)
    19:33:05 LP m1_lora_doc_pos: dev macro re-check 0.5444 (should equal its λ0 score)
    19:39:34 DONE kd_student_m1x6: dev macro λ0 0.5456 (CL 0.4552 IT 0.5885) | λ0.5 0.5300 | 6.5 min
    19:39:34 PLAN DONE
--- GPU 0: last 3 lines
    19:39:29 epoch 6: loss 0.0870 | dev macro-F1 all 0.5314 (CL 0.4425 IT 0.5759) acc 0.578 | 64s
    19:39:34 DONE kd_student_m1x6: dev macro λ0 0.5456 (CL 0.4552 IT 0.5885) | λ0.5 0.5300 | 6.5 min
    19:39:34 PLAN DONE
--- GPU 1: key events
    19:36:31 DONE nokd_student_m1x6: dev macro λ0 0.4586 (CL 0.4089 IT 0.4688) | λ0.5 0.4919 | 6.4 min
    19:36:57 LP m2_lorma_plus: dev macro re-check 0.5262 (should equal its λ0 score)
    19:37:09 LP

## P3-5 · Ensembles, error analysis and test submissions

**P3-4 (KD):** the student (M1 with 6 of 12 layers, 62.4M parameters) distilled from the mean of the 4 best M1 models reaches **0.546** dev macro-F1, against **0.459** for the same student without KD (**+0.087**). It's within 0.01 of the best 100M teacher (0.555), with 38% fewer parameters and about 2× faster test inference (2.2 s against about 4.5 s).

**This cell (CPU only, it uses the saved dev/test log-probs):**
1. Scores every single model and four **ensembles** (mean log-probs) on dev, with label masks and λ ∈ {0, 0.5}.
2. **Error analysis** of the best system: per-role F1 for CL and IT, and the most frequent confusions.
3. Writes **test submissions** (`{doc_id: [role per sentence]}`, `prediction.json` at the root of each zip). For each submission, λ is chosen on dev.

In [22]:
# ============================================================================================
# P3-5 — Ensembles + error analysis + test zips (CPU only, ~2 min; reads p3_lp/*.pkl)
# ============================================================================================
import sys, os, json, glob, pickle, zipfile, importlib
import numpy as np
from collections import Counter
WORK = "/kaggle/working"; sys.path.insert(0, f"{WORK}/src")
import p3_rr as rr; importlib.reload(rr)
from tokenizers import Tokenizer
from sklearn.metrics import f1_score

# ---- 1. data, priors, saved log-probs ----------------------------------------------------------
tk = Tokenizer.from_file(f"{WORK}/tokenizer/hf_a1_bpe_v2_32k.json")
def load_docs(path):
    obj = json.load(open(path))
    if isinstance(obj, dict):
        obj = [{"id": k, **v} if isinstance(v, dict) else {"id": k, "text": v} for k, v in obj.items()]
    return obj
D = {os.path.basename(f)[:-5]: load_docs(f)
     for f in glob.glob(f"{WORK}/data_p3/Phase_3_RR/*/*.json") if not f.endswith("label_vocab.json")}
ENC, KEYS = {}, {"train": [], "dev": [], "test": []}
for dom in ["CL", "IT"]:
    for split in ["train", "dev", "test"]:
        e = rr.encode_docs(D[f"{dom}_{split}"], tk, dom); ENC.update(e); KEYS[split] += list(e)
ALLOWED, TRANS = rr.domain_stats(ENC, KEYS["train"])
LP = {os.path.basename(f)[:-4]: pickle.load(open(f, "rb")) for f in glob.glob(f"{WORK}/p3_lp/*.pkl")}
print("models with saved log-probs:", sorted(LP))

def mix(names, split):                                   # ensemble = mean of log-probs
    return {k: np.mean([LP[n][split][k] for n in names], 0) for k in KEYS[split]}

# ---- 2. dev scores: single models + ensembles, λ = 0 and 0.5 -------------------------------------
TEACH = ["m1_rslora", "m1_lorma_plus", "m1_full", "m1_lora_doc_pos"]
ENS = {"ens_top4":    TEACH,
       "ens_top4_kd": TEACH + ["kd_student_m1x6"],
       "ens_arch":    ["m1_rslora", "m1_lorma_plus", "m2_lorma_plus", "m4_lora"],
       "ens_top7":    TEACH + ["kd_student_m1x6", "m2_lorma_plus", "m4_lora"]}
SYSTEMS = {**{n: [n] for n in sorted(LP)}, **ENS}
SC = {}
for name, members in SYSTEMS.items():
    lp = mix(members, "dev")
    SC[name] = {lam: rr.evaluate(lp, ENC, KEYS["dev"], ALLOWED, TRANS, lam) for lam in (0.0, 0.5)}
print(f"\n{'system':<20}{'λ0 all':>8}{'λ.5 all':>8}{'CL':>8}{'IT':>8}")
for n in sorted(SC, key=lambda n: -max(SC[n][l]['all']['macro_f1'] for l in SC[n])):
    lam = max(SC[n], key=lambda l: SC[n][l]["all"]["macro_f1"])
    print(f"{n:<20}{SC[n][0.0]['all']['macro_f1']:>8.4f}{SC[n][0.5]['all']['macro_f1']:>8.4f}"
          f"{SC[n][lam]['CL']['macro_f1']:>8.4f}{SC[n][lam]['IT']['macro_f1']:>8.4f}")

# ---- 3. error analysis of the best ensemble: per-role F1 per domain + top confusions -----------
best = max(ENS, key=lambda n: max(SC[n][l]["all"]["macro_f1"] for l in SC[n]))
lam_b = max(SC[best], key=lambda l: SC[best][l]["all"]["macro_f1"])
lp = mix(ENS[best], "dev")
G, P = {"CL": [], "IT": []}, {"CL": [], "IT": []}
for k in KEYS["dev"]:
    dom = ENC[k]["dom"]
    G[dom] += ENC[k]["labs"]; P[dom] += rr.decode(lp[k], ALLOWED[dom], TRANS[dom], lam_b)
print(f"\nbest ensemble: {best} (λ={lam_b}) — per-role dev F1 (support in brackets)")
f = {d: f1_score(G[d], P[d], labels=range(rr.NL), average=None, zero_division=0) for d in G}
sup = {d: Counter(G[d]) for d in G}
for i, l in enumerate(rr.LABELS):
    print(f"   {l:<24} CL {f['CL'][i]:.3f} ({sup['CL'][i]:>4})   IT {f['IT'][i]:.3f} ({sup['IT'][i]:>4})")
conf = Counter((rr.LABELS[g], rr.LABELS[p]) for d in G for g, p in zip(G[d], P[d]) if g != p)
print("\nmost frequent confusions (gold -> predicted):")
for (g, p), c in conf.most_common(10):
    print(f"   {g:<24} -> {p:<24} {c}")

# ---- 4. test zips: {orig_doc_id: [role per sentence]}, prediction.json at the zip root ----------
def write_zip(name, members, lam):
    lp = mix(members, "test"); pred = {}
    for k in KEYS["test"]:
        e = ENC[k]
        labs = [rr.LABELS[i] for i in rr.decode(lp[k], ALLOWED[e["dom"]], TRANS[e["dom"]], lam)]
        assert len(labs) == len(e["sents"]), f"{k}: {len(labs)} labels for {len(e['sents'])} sentences"
        pred[e["orig_id"]] = labs
    assert len(pred) == len(KEYS["test"]), "duplicate document ids across CL/IT"
    folder = f"{WORK}/submissions_p3/p3_{name}"; os.makedirs(folder, exist_ok=True)
    json.dump(pred, open(f"{folder}/prediction.json", "w"))
    with zipfile.ZipFile(f"{folder}/p3_{name}.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{folder}/prediction.json", arcname="prediction.json")
    return sum(map(len, pred.values()))

SUBS = [best] + [e for e in ENS if e != best][:1] + [
        "m1_rslora", "m1_lorma_plus", "m2_lorma_plus", "m4_lora", "m3_lora", "kd_student_m1x6",
        "m1_full", "m1_ia3", "m1_prompt", "nokd_student_m1x6", "m1_lora_doc_pos", "m1_lora_sent"]
summary = {}
print("\nsubmissions written:")
for s in SUBS:
    lam = max(SC[s], key=lambda l: SC[s][l]["all"]["macro_f1"])
    n = write_zip(s, SYSTEMS[s], lam)
    summary[s] = {"members": SYSTEMS[s], "lambda": lam, "dev_macro": round(SC[s][lam]["all"]["macro_f1"], 4),
                  "dev_CL": round(SC[s][lam]["CL"]["macro_f1"], 4), "dev_IT": round(SC[s][lam]["IT"]["macro_f1"], 4)}
    print(f"   p3_{s:<20} λ={lam}  dev macro {summary[s]['dev_macro']:.4f}  ({n} sentences)")
json.dump(summary, open(f"{WORK}/stats/p3_submissions.json", "w"), indent=1)
print("\nexample:", next(iter(json.load(open(f"{WORK}/submissions_p3/p3_{best}/prediction.json")).items()))[1][:5])

models with saved log-probs: ['kd_student_m1x6', 'm1_full', 'm1_ia3', 'm1_lora_doc', 'm1_lora_doc_pos', 'm1_lora_sent', 'm1_lorma_plus', 'm1_prompt', 'm1_rslora', 'm2_lorma_plus', 'm3_lora', 'm4_lora', 'nokd_student_m1x6']

system                λ0 all λ.5 all      CL      IT
ens_top7              0.5833  0.5521  0.5059  0.6098
ens_top4_kd           0.5764  0.5400  0.5200  0.6009
ens_arch              0.5745  0.5409  0.5046  0.5835
ens_top4              0.5691  0.5614  0.5067  0.5952
m1_rslora             0.5550  0.5542  0.5192  0.5578
m1_lora_doc_pos       0.5444  0.5520  0.4438  0.5715
m4_lora               0.5231  0.5503  0.4835  0.5603
m1_lorma_plus         0.5503  0.5472  0.4934  0.5571
m1_full               0.5468  0.5297  0.4622  0.5780
kd_student_m1x6       0.5456  0.5300  0.4552  0.5885
m1_lora_doc           0.5318  0.5326  0.4377  0.5355
m2_lorma_plus         0.5262  0.5298  0.4104  0.5359
m1_ia3                0.4996  0.4960  0.4364  0.5471
nokd_student_m1x6     0.4586  0.49

In [23]:
# ============================================================================================
# P3-B — Backup incl. Phase 3 to the private HF repo + verification
#   Adds data_p3/, p3_ckpt/, p3_lp/, submissions_p3/ to backup_hf.py's allow-list (only once);
#   src/** and stats/** are already covered (p3_rr.py, p3_run.py, p3_results.json, p3_runs/, ...).
# ============================================================================================
import os, subprocess
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi

WORK = "/kaggle/working"
p = f"{WORK}/src/backup_hf.py"

# ---- 1. extend the allow-list (idempotent) -------------------------------------------------------
s = open(p).read()
if "p3_ckpt" not in s:
    s = s.replace('"submissions_p2/**",',
                  '"submissions_p2/**", "data_p3/**", "p3_ckpt/**", "p3_lp/**", "submissions_p3/**",')
    open(p, "w").write(s)
print("Phase-3 folders in allow-list:", "p3_ckpt" in open(p).read())

# ---- 2. run the backup (token via environment only) -------------------------------------------
env = {**os.environ, "HF_WRITE_TOKEN": UserSecretsClient().get_secret("HF_WRITE_TOKEN")}
r = subprocess.run(f"python {p}", shell=True, capture_output=True, text=True, env=env)
print(r.stdout[-600:])
if r.returncode != 0:
    print(r.stderr[-1500:])

# ---- 3. verify: count Phase-3 files now on the hub -------------------------------------------
api = HfApi(token=env["HF_WRITE_TOKEN"])
files = api.list_repo_files(f"{api.whoami()['name']}/cs781-phase1-work")
for pre in ["src/p3_", "data_p3/", "p3_ckpt/", "p3_lp/", "submissions_p3/", "stats/p3_"]:
    hit = [f for f in files if f.startswith(pre)]
    print(f"  {pre:<16} {len(hit):3d} files", ("  e.g. " + hit[0]) if hit else "")

Phase-3 folders in allow-list: True
BACKUP DONE -> Utkarsh-1718-1/cs781-phase1-work in 33s

  src/p3_            2 files   e.g. src/p3_rr.py
  data_p3/           7 files   e.g. data_p3/Phase_3_RR/Test/CL_test.json
  p3_ckpt/          13 files   e.g. p3_ckpt/kd_student_m1x6.pt
  p3_lp/            13 files   e.g. p3_lp/kd_student_m1x6.pkl
  submissions_p3/   28 files   e.g. submissions_p3/p3_ens_top4/p3_ens_top4.zip
  stats/p3_          8 files   e.g. stats/p3_gpu0.log


## P3-X · Phase 3 upgrades (document BiGRU, train+dev retraining, extra seeds)
Document-level BiGRU (h_*): freeze the best fine-tuned models and take each sentence’s vector for the whole judgment. A 2-layer BiGRU then reads the full sentence sequence, so every sentence sees the entire document, not just its 512-token window. This is the same idea that won Phases 4 and 5. Trained on train, best epoch chosen on dev.
Train + dev retraining (*_td): Phase 3’s dev set is small and test came out about 0.06 below dev, so the models are short of data. Each top configuration is retrained on train + dev for the number of epochs where it peaked on dev. There’s no dev left to check these, so they reuse the decisions of their dev-validated versions.
Extra seeds (*_s1): cheap ensemble diversity.

P3-X3 compares every new system with the submitted ens_top7 (0.518 test) on dev, and writes each system plus its train+dev version as a zip.

In [1]:
# ---- P3-X0: make sure everything Phase 3 needs is in this session (restore missing parts from HF) ----
import os
WORK, REPO = "/kaggle/working", "Utkarsh-1718-1/cs781-phase1-work"
need = {"src/p3_rr.py": "code", "tokenizer/hf_a1_bpe_v2_32k.json": "tokenizer",
        "checkpoints/m1_dense_100m/best.pt": "M1", "checkpoints/m2_mla_moe/best.pt": "M2",
        "checkpoints/m4_copy/best.pt": "M4", "p3_ckpt/m1_rslora.pt": "Phase-3 weights",
        "p3_lp/m1_rslora.pkl": "Phase-3 log-probs", "stats/p3_runs/m1_rslora.json": "Phase-3 results"}
missing = [p for p in need if not os.path.exists(f"{WORK}/{p}")]
if not os.path.isdir(f"{WORK}/data_p3/Phase_3_RR"):
    missing.append("data_p3")
print("missing:", missing or "nothing")
if missing:                                                          # download only what Phase 3 needs
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import snapshot_download
    snapshot_download(REPO, repo_type="model", local_dir=WORK, token=UserSecretsClient().get_secret("HF_WRITE_TOKEN"),
                      allow_patterns=["src/**", "tokenizer/**", "data_p3/**", "p3_ckpt/**", "p3_lp/**", "stats/**",
                                      "checkpoints/m1_dense_100m/best.pt", "checkpoints/m2_mla_moe/best.pt",
                                      "checkpoints/m3_encdec/best.pt", "checkpoints/m4_copy/best.pt"])
    print("still missing:", [p for p in need if not os.path.exists(f"{WORK}/{p}")] or "nothing ✅")
import torch; print("GPUs:", torch.cuda.device_count())

missing: ['stats/p3_runs/m1_rslora.json']


Fetching 180 files:   0%|          | 0/180 [00:00<?, ?it/s]

still missing: ['stats/p3_runs/m1_rslora.json']
GPUs: 2


In [2]:
%%writefile /kaggle/working/src/p3_extra.py
"""
p3_extra.py -- Phase-3 upgrades, run as a BACKGROUND process (one per GPU), same pattern as p3_run.py:

    CUDA_VISIBLE_DEVICES=0 PYTHONUNBUFFERED=1 setsid nohup python p3_extra.py --plan plan.json >> log 2>&1 &
    python p3_extra.py --smoke          # 1-2 min end-to-end check of every code path (run this first)

Job types (plan.json = list of jobs):
  {"type": "hier", "name": N, "bases": [run, ...]}
        document-level model: per-sentence embeddings of the (frozen) fine-tuned runs `bases` for the
        WHOLE judgment -> 2-layer BiGRU over the sentence sequence -> 13 roles. Trained on train,
        best epoch chosen on dev. Embeddings are cached in p3_emb/<run>.pkl.
  {"type": "td", "name": X}
        retrain run X's exact config on TRAIN + DEV for X's best epoch count (no held-out data, so
        the last epoch is kept). Output X_td (test log-probs only).
  {"type": "hier_td", "name": N_td, "ref": N}
        hierarchical model N rebuilt on the train+dev versions of its bases (<base>_td), trained on
        train + dev for N's best epoch count.
  {"type": "seed", "name": X, "seed": s}
        rerun X's config with another seed (train only, dev selection) -> X_s<s> (ensemble diversity).
Outputs follow p3_run.py: stats/p3_runs/<name>.json (written LAST = done marker), p3_ckpt/<name>.pt,
p3_lp/<name>.pkl ({"dev", "test"} log-probs per document; td runs: {"test"} only).
"""
import argparse
import glob
import json
import os
import pickle
import random
import sys
import time
import traceback

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

WORK = "/kaggle/working"
sys.path.insert(0, f"{WORK}/src")
import p3_rr as rr                                                    # noqa: E402
import p2_peft as pp                                                  # noqa: E402
from tokenizers import Tokenizer                                     # noqa: E402

ap = argparse.ArgumentParser()
ap.add_argument("--plan"); ap.add_argument("--smoke", action="store_true")
ARGS = ap.parse_args()

DEV = "cuda:0" if torch.cuda.is_available() else "cpu"
CK = {"m1": f"{WORK}/checkpoints/m1_dense_100m/best.pt", "m2": f"{WORK}/checkpoints/m2_mla_moe/best.pt",
      "m3": f"{WORK}/checkpoints/m3_encdec/best.pt", "m4": f"{WORK}/checkpoints/m4_copy/best.pt"}
OUT, LPD, CKD, EMB = f"{WORK}/stats/p3_runs", f"{WORK}/p3_lp", f"{WORK}/p3_ckpt", f"{WORK}/p3_emb"
for d in (OUT, LPD, CKD, EMB):
    os.makedirs(d, exist_ok=True)


def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)


# ---- data: identical encoding / priors / class weights to p3_run.py ------------------------------
def load_docs(path):
    obj = json.load(open(path))
    if isinstance(obj, dict):
        obj = [{"id": k, **v} if isinstance(v, dict) else {"id": k, "text": v} for k, v in obj.items()]
    return obj


tk = Tokenizer.from_file(f"{WORK}/tokenizer/hf_a1_bpe_v2_32k.json")
D = {os.path.basename(f)[:-5]: load_docs(f)
     for f in glob.glob(f"{WORK}/data_p3/Phase_3_RR/*/*.json") if not f.endswith("label_vocab.json")}
ENC, KEYS = {}, {"train": [], "dev": [], "test": []}
for dom in ["CL", "IT"]:
    for split in ["train", "dev", "test"]:
        e = rr.encode_docs(D[f"{dom}_{split}"], tk, dom)
        ENC.update(e); KEYS[split] += list(e)
KEYS["td"] = KEYS["train"] + KEYS["dev"]
ALLOWED, TRANS = rr.domain_stats(ENC, KEYS["train"])


def class_weights(keys):
    cnt = np.bincount([l for k in keys for l in ENC[k]["labs"]], minlength=rr.NL).astype(float)
    return torch.tensor(np.where(cnt > 0, np.sqrt(cnt.max() / np.maximum(cnt, 1)), 1.0),
                        dtype=torch.float).clamp(max=10.0)


CW, CW_TD = class_weights(KEYS["train"]), class_weights(KEYS["td"])
OLD = json.load(open(f"{WORK}/stats/p3_results.json")) if os.path.exists(f"{WORK}/stats/p3_results.json") else {}
log(f"data ready: train {len(KEYS['train'])} | dev {len(KEYS['dev'])} | test {len(KEYS['test'])} docs | device {DEV}")


# ---- helpers ------------------------------------------------------------------------------------
def result(name):
    """Result record of a finished run (p3_run.py json, or the old notebook results file)."""
    p = f"{OUT}/{name}.json"
    return json.load(open(p)) if os.path.exists(p) else OLD[name]


def best_epoch(R):
    h = R.get("hist") or []
    if h and isinstance(h[0], dict) and "all" in h[0]:
        return int(max(h, key=lambda x: x["all"])["epoch"])
    return int((R.get("cfg") or {}).get("epochs") or 6)


def build_from(cfg):
    """Fresh model (pretrained backbone + method) from a run config; returns (model, ctx, W)."""
    c = {k: v for k, v in cfg.items() if v is not None}
    ctx, W, ckpt, method = c.pop("ctx"), c.pop("W"), c.pop("ckpt"), c.pop("method")
    bidir, use_pos, layers = c.pop("bidir"), c.pop("use_pos"), c.pop("student_layers", None)
    for k in ("epochs", "lr", "seed", "bs", "teachers", "kd_T", "kd_lam"):
        c.pop(k, None)
    m = rr.build_rr(CK[ckpt], method, bidirectional=bidir, use_pos=use_pos, **c)
    if layers:
        pp.shrink_dense(m.tagger, tuple(layers))
    return m, ctx, W


def lam_report(LPd):
    out = {}
    for lam in (0.0, 0.5, 1.0):
        r = rr.evaluate(LPd, ENC, KEYS["dev"], ALLOWED, TRANS, lam)
        out[f"lam{lam}"] = {k: {m: round(v, 4) for m, v in d.items()} for k, d in r.items()}
    return out


# ---- job 1: retrain on train + dev ---------------------------------------------------------------
def td_job(base):
    t0, name = time.time(), f"{base}_td"
    R = result(base); c = dict(R["cfg"])
    if c.get("teachers"):
        raise ValueError(f"{base} is a KD run; not retrained on train+dev")
    ep = best_epoch(R); lr = c.get("lr") or 5e-4; seed = c.get("seed") or 0
    m, ctx, W = build_from(c)
    bs = c.get("bs") or (32 if ctx == "none" else 8)
    ex = rr.make_examples(ENC, KEYS["td"], ctx, W)
    n = {"e": 0}

    def ev(_):                                                        # no held-out data: a counter makes
        n["e"] += 1; r = {"macro_f1": float(n["e"]), "acc": 0.0}      # train_rr keep the LAST epoch
        return {"all": r, "CL": r, "IT": r}
    log(f"=== {name}: {base}'s config on train+dev ({len(KEYS['td'])} docs) for {ep} epochs, lr {lr} "
        f"('dev macro-F1' in this log = epoch counter, not a score)")
    rr.train_rr(m, ex, ev, epochs=ep, lr=lr, bs=bs, class_weight=CW_TD, device=DEV, seed=seed, log=log)
    LPt = rr.predict(m, ENC, KEYS["test"], ctx, W, DEV)
    rr.save_trainable(m, f"{CKD}/{name}.pt")
    pickle.dump({"test": LPt}, open(f"{LPD}/{name}.pkl", "wb"))
    json.dump({"cfg": R["cfg"], "base": base, "td": True, "epochs_trained": ep,
               "min": round((time.time() - t0) / 60, 1)}, open(f"{OUT}/{name}.json", "w"), indent=1)
    log(f"DONE {name}: {ep} epochs on train+dev | {round((time.time() - t0) / 60, 1)} min")
    del m; torch.cuda.empty_cache()


# ---- job 2: another seed -----------------------------------------------------------------------
def seed_job(base, seed):
    t0, name = time.time(), f"{base}_s{seed}"
    R = result(base); c = dict(R["cfg"])
    ep = int(c.get("epochs") or 6); lr = c.get("lr") or 5e-4
    m, ctx, W = build_from(c)
    bs = c.get("bs") or (32 if ctx == "none" else 8)
    ex = rr.make_examples(ENC, KEYS["train"], ctx, W)
    ev = lambda mm: rr.evaluate(rr.predict(mm, ENC, KEYS["dev"], ctx, W, DEV), ENC, KEYS["dev"], ALLOWED)
    log(f"=== {name}: {base}'s config, seed {seed}, {ep} epochs")
    hist, _ = rr.train_rr(m, ex, ev, epochs=ep, lr=lr, bs=bs, class_weight=CW, device=DEV, seed=seed, log=log)
    LPd = rr.predict(m, ENC, KEYS["dev"], ctx, W, DEV); LPt = rr.predict(m, ENC, KEYS["test"], ctx, W, DEV)
    rr.save_trainable(m, f"{CKD}/{name}.pt")
    pickle.dump({"dev": LPd, "test": LPt}, open(f"{LPD}/{name}.pkl", "wb"))
    out = {"cfg": {**R["cfg"], "seed": seed}, "base": base, "hist": hist,
           "min": round((time.time() - t0) / 60, 1), **lam_report(LPd)}
    json.dump(out, open(f"{OUT}/{name}.json", "w"), indent=1)
    log(f"DONE {name}: dev macro λ0 {out['lam0.0']['all']['macro_f1']:.4f} | λ0.5 {out['lam0.5']['all']['macro_f1']:.4f} "
        f"| {out['min']} min")
    del m; torch.cuda.empty_cache()


# ---- job 3: document-level BiGRU over sentence embeddings ----------------------------------------
@torch.no_grad()
def sent_emb(m, keys, ctx, W, bs=16):
    """Per-sentence vector the role head sees (mean-pooled tokens + position), averaged over windows."""
    m.eval(); buf = {}
    hook = m.drop.register_forward_hook(lambda mod, inp, out: buf.__setitem__("x", inp[0].detach().float()))
    ex = sorted(rr.make_examples(ENC, keys, ctx, W), key=lambda x: len(x["ids"]))
    d = m.tagger.backbone.cfg.d_model
    acc = {k: np.zeros((len(ENC[k]["sents"]), d), np.float32) for k in keys}
    cnt = {k: np.zeros(len(ENC[k]["sents"])) for k in keys}
    for b in range(0, len(ex), bs):
        chunk = ex[b:b + bs]
        ids, seg, off, _, pos = rr.collate(chunk, DEV)
        with rr._autocast(DEV):
            m(ids, seg, off, pos)
        x, j = buf["x"].cpu().numpy(), 0
        for e in chunk:
            for i in e["sidx"]:
                acc[e["key"]][i] += x[j]; cnt[e["key"]][i] += 1; j += 1
    hook.remove()
    return {k: (acc[k] / cnt[k][:, None]).astype(np.float16) for k in keys}


def get_emb(run, splits=("train", "dev", "test")):
    path = f"{EMB}/{run}.pkl"
    if os.path.exists(path):
        return pickle.load(open(path, "rb"))
    src = run[:-3] if run.endswith("_td") else run
    m, ctx, W = build_from(result(src)["cfg"])
    rr.load_trainable(m, f"{CKD}/{run}.pt"); m.to(DEV)
    E = {}
    for s in splits:
        ti = time.time(); E[s] = sent_emb(m, KEYS[s], ctx, W)
        log(f"   embeddings {run} / {s}: {sum(len(v) for v in E[s].values())} sentences in {time.time()-ti:.0f}s")
    pickle.dump(E, open(path, "wb")); del m; torch.cuda.empty_cache()
    return E


class DocGRU(nn.Module):
    """Sentence embeddings of a whole judgment -> 2-layer BiGRU -> role logits for every sentence."""
    def __init__(self, d, h=256, p=0.3):
        super().__init__()
        self.inp = nn.Sequential(nn.LayerNorm(d), nn.Dropout(p))
        self.dom = nn.Embedding(2, d)                                 # CL / IT
        self.gru = nn.GRU(d, h, num_layers=2, batch_first=True, bidirectional=True, dropout=p)
        self.out = nn.Sequential(nn.Dropout(p), nn.Linear(2 * h, rr.NL))

    def forward(self, x, dom):                                        # x (B, T, d), dom (B,)
        z, _ = self.gru(self.inp(x) + self.dom(dom)[:, None])
        return self.out(z)


def _feats(Es, k):
    split = "test" if k in TEST_SET else ("dev" if k in DEV_SET else "train")
    return np.concatenate([E[split][k] for E in Es], 1).astype(np.float32)


def _batch(Es, keys):
    X = [_feats(Es, k) for k in keys]; T = max(len(x) for x in X)
    x = np.zeros((len(keys), T, X[0].shape[1]), np.float32); y = np.full((len(keys), T), -100, np.int64)
    for b, (k, f) in enumerate(zip(keys, X)):
        x[b, :len(f)] = f; y[b, :len(f)] = ENC[k]["labs"][:len(f)] if ENC[k]["labs"][0] != -100 else -100
    dom = torch.tensor([0 if ENC[k]["dom"] == "CL" else 1 for k in keys], device=DEV)
    return torch.from_numpy(x).to(DEV), torch.from_numpy(y).to(DEV), dom


@torch.no_grad()
def _gru_lp(net, Es, keys):
    net.eval(); out = {}
    for b in range(0, len(keys), 8):
        ks = keys[b:b + 8]; x, _, dom = _batch(Es, ks)
        lp = net(x, dom).float().log_softmax(-1).cpu().numpy()
        for j, k in enumerate(ks):
            out[k] = lp[j, :len(ENC[k]["sents"])]
    return out


TEST_SET, DEV_SET = set(KEYS["test"]), set(KEYS["dev"])


def train_gru(Es, train_keys, epochs, cw, seed=0, select_on_dev=True, lr=1e-3, bs=4):
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
    net = DocGRU(_feats(Es, train_keys[0]).shape[1]).to(DEV)
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=0.01)
    steps = epochs * int(np.ceil(len(train_keys) / bs))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.1)
    best, best_state, best_ep, hist = -1.0, None, epochs, []
    for ep in range(epochs):
        net.train(); order = list(train_keys); random.shuffle(order); tot = 0.0
        for b in range(0, len(order), bs):
            x, y, dom = _batch(Es, order[b:b + bs])
            loss = F.cross_entropy(net(x, dom).reshape(-1, rr.NL), y.reshape(-1), weight=cw.to(DEV), ignore_index=-100)
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step(); sched.step(); tot += loss.item()
        if select_on_dev:
            LPd = _gru_lp(net, Es, KEYS["dev"])
            f = max(rr.evaluate(LPd, ENC, KEYS["dev"], ALLOWED, TRANS, lam)["all"]["macro_f1"] for lam in (0.0, 0.5))
            hist.append({"epoch": ep + 1, "loss": tot, "all": f})
            if f > best:
                best, best_ep = f, ep + 1
                best_state = {k: v.detach().clone() for k, v in net.state_dict().items()}
    if best_state is not None:
        net.load_state_dict(best_state)
    return net, hist, best_ep


def hier_job(name, bases, epochs=30, seed=0):
    t0 = time.time(); Es = [get_emb(b) for b in bases]
    log(f"=== {name}: document BiGRU over sentence embeddings of {bases} ({Es[0]['dev'][KEYS['dev'][0]].shape[1] * len(Es)}-d)")
    net, hist, best_ep = train_gru(Es, KEYS["train"], epochs, CW, seed=seed)
    LPd, LPt = _gru_lp(net, Es, KEYS["dev"]), _gru_lp(net, Es, KEYS["test"])
    out = {"cfg": {"type": "hier", "bases": bases, "epochs": epochs, "seed": seed}, "best_epoch": best_ep,
           "hist": hist, "min": round((time.time() - t0) / 60, 1), **lam_report(LPd)}
    torch.save(net.state_dict(), f"{CKD}/{name}.pt")
    pickle.dump({"dev": LPd, "test": LPt}, open(f"{LPD}/{name}.pkl", "wb"))
    json.dump(out, open(f"{OUT}/{name}.json", "w"), indent=1)
    log(f"DONE {name}: dev macro λ0 {out['lam0.0']['all']['macro_f1']:.4f} | λ0.5 {out['lam0.5']['all']['macro_f1']:.4f} "
        f"(best epoch {best_ep}) | {out['min']} min")


def hier_td_job(name, ref):
    t0 = time.time(); R = result(ref); bases = [b + "_td" for b in R["cfg"]["bases"]]
    missing = [b for b in bases if not os.path.exists(f"{CKD}/{b}.pt")]
    if missing:
        raise FileNotFoundError(f"train+dev bases not finished yet: {missing}")
    Es = [get_emb(b) for b in bases]
    log(f"=== {name}: {ref}'s BiGRU on train+dev bases {bases} for {R['best_epoch']} epochs")
    net, _, _ = train_gru(Es, KEYS["td"], R["best_epoch"], CW_TD, seed=R["cfg"].get("seed", 0), select_on_dev=False)
    LPt = _gru_lp(net, Es, KEYS["test"])
    torch.save(net.state_dict(), f"{CKD}/{name}.pt")
    pickle.dump({"test": LPt}, open(f"{LPD}/{name}.pkl", "wb"))
    json.dump({"cfg": {"type": "hier_td", "ref": ref, "bases": bases}, "td": True, "epochs_trained": R["best_epoch"],
               "min": round((time.time() - t0) / 60, 1)}, open(f"{OUT}/{name}.json", "w"), indent=1)
    log(f"DONE {name} | {round((time.time() - t0) / 60, 1)} min")


# ---- smoke test: every code path on a few documents, nothing written to the real outputs -----------
if ARGS.smoke:
    base = "m1_rslora"
    m, ctx, W = build_from(result(base)["cfg"]); rr.load_trainable(m, f"{CKD}/{base}.pt"); m.to(DEV)
    ks = KEYS["train"][:3] + KEYS["dev"][:2] + KEYS["test"][:1]
    E = {s: sent_emb(m, [k for k in ks if (k in DEV_SET) == (s == "dev") and (k in TEST_SET) == (s == "test")], ctx, W)
         for s in ("train", "dev", "test")}
    log("embeddings ok:", {s: len(v) for s, v in E.items()}, "dim", next(iter(E["dev"].values())).shape)
    tr_keys = list(E["train"]); KEYS_DEV_BAK = KEYS["dev"]; KEYS["dev"] = list(E["dev"])
    net, hist, be = train_gru([E], tr_keys, 2, CW)
    log("BiGRU ok: hist", hist, "| test lp shape", next(iter(_gru_lp(net, [E], list(E["test"])).values())).shape)
    KEYS["dev"] = KEYS_DEV_BAK
    log("td-config ok: best epoch of", base, "=", best_epoch(result(base)), "| cfg", result(base)["cfg"])
    log("SMOKE TEST PASSED"); sys.exit(0)


PLAN = json.load(open(ARGS.plan))
for job in PLAN:
    job = dict(job); name = job.pop("name"); typ = job.pop("type")
    tgt = {"td": f"{name}_td", "seed": f"{name}_s{job.get('seed')}"}.get(typ, name)
    if os.path.exists(f"{OUT}/{tgt}.json"):
        log(f"skip {tgt} (already done)"); continue
    try:
        if typ == "td":
            td_job(name)
        elif typ == "seed":
            seed_job(name, job["seed"])
        elif typ == "hier":
            hier_job(name, **job)
        elif typ == "hier_td":
            hier_td_job(name, **job)
    except Exception:
        log(f"!!! {tgt} FAILED"); traceback.print_exc(); torch.cuda.empty_cache()
log("PLAN DONE")

Writing /kaggle/working/src/p3_extra.py


In [3]:
# ---- P3-X1b: smoke test (~2 min): embeddings, BiGRU, configs on a few documents. Must end with SMOKE TEST PASSED ----
!cd /kaggle/working/src && python p3_extra.py --smoke 2>&1 | tail -n 15

08:11:57 data ready: train 80 | dev 10 | test 10 docs | device cuda:0
08:12:03 embeddings ok: {'train': 3, 'dev': 2, 'test': 1} dim (265, 768)
08:12:08 BiGRU ok: hist [{'epoch': 1, 'loss': 2.606300115585327, 'all': 0.03648347601951579}, {'epoch': 2, 'loss': 1.7006285190582275, 'all': 0.03648347601951579}] | test lp shape (18, 13)
08:12:08 td-config ok: best epoch of m1_rslora = 3 | cfg {'ckpt': 'm1', 'method': 'rslora', 'ctx': 'doc', 'W': 512, 'use_pos': True, 'bidir': True, 'epochs': 6, 'lr': 0.0002, 'seed': 0, 'rs_r': 16}
08:12:08 SMOKE TEST PASSED


In [4]:
# ---- P3-X2: launch both GPU queues, stay busy until they finish (keep-alive), then back up ----
import json, os, subprocess, time
from IPython.display import clear_output
WORK = "/kaggle/working"
sh = lambda c: subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip()
assert __import__("torch").cuda.device_count() == 2, "turn on the T4 x2 accelerator first"

TOP3 = ["m1_rslora", "m1_full", "m1_lorma_plus"]                     # best three Phase-3 models on dev
plans = {
    0: [dict(type="hier", name="h_rslora", bases=["m1_rslora"]),           # 1) document BiGRU (train only)
        dict(type="hier", name="h_cat3", bases=TOP3),
        *[dict(type="td", name=b) for b in TOP3],                          # 2) retrain on train+dev
        dict(type="hier_td", name="h_cat3_td", ref="h_cat3"),              #    BiGRU on train+dev bases
        dict(type="hier_td", name="h_rslora_td", ref="h_rslora")],
    1: [*[dict(type="td", name=b) for b in ("m1_lora_doc_pos", "m2_lorma_plus", "m4_lora")],
        *[dict(type="seed", name=b, seed=1) for b in TOP3]],               # 3) extra seeds
}
for g, plan in plans.items():
    if sh(f"pgrep -f '[p]3_extra.py --plan .*p3x_gpu{g}.json'"):
        print(f"GPU {g}: already running"); continue
    pf = f"{WORK}/stats/p3x_gpu{g}.json"; json.dump(plan, open(pf, "w"), indent=1)
    subprocess.run(f"cd {WORK}/src && CUDA_VISIBLE_DEVICES={g} PYTHONUNBUFFERED=1 setsid nohup python p3_extra.py "
                   f"--plan {pf} >> {WORK}/stats/p3x_gpu{g}.log 2>&1 < /dev/null &", shell=True)
    print(f"GPU {g}: launched {[j['name'] + {'td': '_td', 'seed': '_s1'}.get(j['type'], '') for j in plan]}")

time.sleep(30)
while sh("pgrep -f '[p]3_extra.py --plan'"):                       # keep the session busy while training
    clear_output(wait=True)
    print(f"[{time.strftime('%H:%M', time.gmtime(time.time() + 19800))} IST]")
    for g in (0, 1):
        print(f"GPU {g}: {sh(f'grep -c DONE {WORK}/stats/p3x_gpu{g}.log')} done | "
              f"{sh(f'grep -c FAILED {WORK}/stats/p3x_gpu{g}.log')} failed | {sh(f'tail -n 1 {WORK}/stats/p3x_gpu{g}.log')[:130]}")
    time.sleep(120)
print(sh(f"grep -h -E 'DONE|FAILED' {WORK}/stats/p3x_gpu0.log {WORK}/stats/p3x_gpu1.log"))

from kaggle_secrets import UserSecretsClient                        # back up before anything else happens
from huggingface_hub import HfApi
HfApi(token=UserSecretsClient().get_secret("HF_WRITE_TOKEN")).upload_folder(
    repo_id="Utkarsh-1718-1/cs781-phase1-work", repo_type="model", folder_path=WORK,
    commit_message="phase 3 upgrades", ignore_patterns=["**/__pycache__/**"],
    allow_patterns=["p3_ckpt/**", "p3_lp/**", "p3_emb/**", "src/**", "stats/**"])
print("backup done ✅ -> run P3-X3")

[14:32 IST]
GPU 0: 8 done | 0 failed | 08:33:33 PLAN DONE
GPU 1: 5 done | 0 failed | 09:02:07 epoch 5: loss 0.3327 | dev macro-F1 all 0.5300 (CL 0.4659 IT 0.5239) acc 0.587 | 91s


It seems you are trying to upload a large folder at once. This might take some time and then fail if the folder is too large. For such cases, it is recommended to upload in smaller batches or to use `HfApi().upload_large_folder(...)`/`hf upload-large-folder` instead. For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/upload#upload-a-large-folder.


08:13:39 DONE h_rslora: dev macro λ0 0.5415 | λ0.5 0.6019 (best epoch 19) | 1.4 min
08:15:56 DONE h_cat3: dev macro λ0 0.5544 | λ0.5 0.5314 (best epoch 16) | 2.3 min
08:21:03 DONE m1_rslora_td: 3 epochs on train+dev | 5.1 min
08:25:31 DONE m1_full_td: 2 epochs on train+dev | 4.5 min
08:30:29 DONE m1_lorma_plus_td: 3 epochs on train+dev | 5.0 min
08:33:07 DONE h_cat3_td | 2.6 min
08:33:33 DONE h_rslora_td | 0.4 min
08:33:33 PLAN DONE
08:17:21 DONE m1_lora_doc_pos_td: 3 epochs on train+dev | 5.1 min
08:29:10 DONE m2_lorma_plus_td: 5 epochs on train+dev | 11.8 min
08:32:38 DONE m4_lora_td: 2 epochs on train+dev | 3.5 min
08:42:15 DONE m1_rslora_s1: dev macro λ0 0.5379 | λ0.5 0.5222 | 9.6 min
08:54:26 DONE m1_full_s1: dev macro λ0 0.5416 | λ0.5 0.5279 | 12.2 min
09:03:47 DONE m1_lorma_plus_s1: dev macro λ0 0.5306 | λ0.5 0.5373 | 9.3 min
09:03:47 PLAN DONE


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

backup done ✅ -> run P3-X3


In [5]:
# ---- P3-X3: new ensembles + train+dev counterparts -> Phase-3 zips (dev = model selection only) ----
import sys, os, re, json, glob, pickle, zipfile, importlib
import numpy as np
WORK = "/kaggle/working"; sys.path.insert(0, f"{WORK}/src")
import p3_rr as rr; importlib.reload(rr)
from tokenizers import Tokenizer

# 1) data + priors exactly as in the earlier Phase-3 cells (train priors for dev scoring,
#    train+dev priors for the train+dev systems)
tk = Tokenizer.from_file(f"{WORK}/tokenizer/hf_a1_bpe_v2_32k.json")
def load_docs(path):
    obj = json.load(open(path))
    if isinstance(obj, dict):
        obj = [{"id": k, **v} if isinstance(v, dict) else {"id": k, "text": v} for k, v in obj.items()]
    return obj
D = {os.path.basename(f)[:-5]: load_docs(f)
     for f in glob.glob(f"{WORK}/data_p3/Phase_3_RR/*/*.json") if not f.endswith("label_vocab.json")}
ENC, KEYS = {}, {"train": [], "dev": [], "test": []}
for dom in ["CL", "IT"]:
    for split in ["train", "dev", "test"]:
        e = rr.encode_docs(D[f"{dom}_{split}"], tk, dom); ENC.update(e); KEYS[split] += list(e)
ALLOWED, TRANS = rr.domain_stats(ENC, KEYS["train"])
ALLOWED_TD, TRANS_TD = rr.domain_stats(ENC, KEYS["train"] + KEYS["dev"])
LP = {os.path.basename(f)[:-4]: pickle.load(open(f, "rb")) for f in glob.glob(f"{WORK}/p3_lp/*.pkl")}
mix = lambda names, split: {k: np.mean([LP[n][split][k] for n in names], 0) for k in KEYS[split]}

# 2) candidate systems (all members must have dev log-probs)
has_dev = {n for n in LP if "dev" in LP[n]}
TOP7 = ["m1_rslora", "m1_lorma_plus", "m1_full", "m1_lora_doc_pos", "kd_student_m1x6", "m2_lorma_plus", "m4_lora"]
HIER = sorted(n for n in has_dev if n.startswith("h_"))                       # document BiGRU models
SEEDS = sorted(n for n in has_dev if re.search(r"_s\d+$", n))                 # extra seeds
SYSTEMS = {"ens_top7": TOP7, "ens_top7_seeds": TOP7 + SEEDS, "ens_top7_hier": TOP7 + HIER,
           "ens_all": TOP7 + SEEDS + HIER, "ens_hier": HIER, **{n: [n] for n in HIER + SEEDS}}
SYSTEMS = {k: [m for m in v if m in has_dev] for k, v in SYSTEMS.items()}
SYSTEMS = {k: v for k, v in SYSTEMS.items() if v}

# 3) dev score of every system at Viterbi weight lambda in {0, 0.5, 1}; keep the best lambda
SC = {}
for name, mem in SYSTEMS.items():
    lp = mix(mem, "dev")
    r = {lam: rr.evaluate(lp, ENC, KEYS["dev"], ALLOWED, TRANS, lam)["all"]["macro_f1"] for lam in (0.0, 0.5, 1.0)}
    lam = max(r, key=r.get); SC[name] = {"members": mem, "lambda": lam, "dev": round(r[lam], 4)}
print(f"{'system':<24}{'dev macro-F1':>13}{'λ':>5}  members")
for n in sorted(SC, key=lambda n: -SC[n]["dev"]):
    print(f"{n:<24}{SC[n]['dev']:>13.4f}{SC[n]['lambda']:>5}  {SC[n]['members']}")

# 4) zips: {orig_doc_id: [role per sentence]} with prediction.json at the zip root (as before)
def write_zip(name, members, lam, allowed, trans):
    lp = mix(members, "test"); pred = {}
    for k in KEYS["test"]:
        e = ENC[k]
        labs = [rr.LABELS[i] for i in rr.decode(lp[k], allowed[e["dom"]], trans[e["dom"]], lam)]
        assert len(labs) == len(e["sents"]), f"{k}: {len(labs)} labels for {len(e['sents'])} sentences"
        pred[e["orig_id"]] = labs
    assert len(pred) == len(KEYS["test"]), "duplicate document ids across CL/IT"
    folder = f"{WORK}/submissions_p3/p3_{name}"; os.makedirs(folder, exist_ok=True)
    json.dump(pred, open(f"{folder}/prediction.json", "w"))
    with zipfile.ZipFile(f"{folder}/p3_{name}.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{folder}/prediction.json", arcname="prediction.json")
    return sum(map(len, pred.values()))

# train+dev counterpart: every member replaced by its _td version when one exists (same lambda)
td_of = lambda mem: [m + "_td" if m + "_td" in LP else m for m in mem]
rows = []
for n in sorted(SC, key=lambda n: -SC[n]["dev"]):
    if n == "ens_top7":
        continue                                                              # already submitted (0.518 test)
    s = SC[n]; k = write_zip(n, s["members"], s["lambda"], ALLOWED, TRANS)
    rows.append((n, s["dev"], "train only", k))
for n in sorted(SC, key=lambda n: -SC[n]["dev"]):
    s = SC[n]; mem = td_of(s["members"]); n_td = sum(m.endswith("_td") for m in mem)
    if n_td:
        k = write_zip(n + "_td", mem, s["lambda"], ALLOWED_TD, TRANS_TD)
        rows.append((n + "_td", s["dev"], f"train+dev ({n_td}/{len(mem)} members)", k))
print(f"\n{'zip':<30}{'dev of its system':>18}  trained on")
for n, d, how, k in rows:
    print(f"p3_{n:<27}{d:>18.4f}  {how}  [{k} sentences]")
json.dump({"systems": SC, "zips": rows}, open(f"{WORK}/stats/p3_upgrade.json", "w"), indent=1)
best = max(SC, key=lambda n: SC[n]["dev"])
print(f"\nbest dev system: {best} ({SC[best]['dev']:.4f}) vs ens_top7 {SC.get('ens_top7', {}).get('dev')}")
print(f"upload first: p3_{best}_td (if written), then p3_{best}, then p3_ens_top7_td")

system                   dev macro-F1    λ  members
h_rslora                       0.6019  0.5  ['h_rslora']
ens_top7_hier                  0.5836  0.0  ['m1_rslora', 'm1_lorma_plus', 'm1_full', 'm1_lora_doc_pos', 'kd_student_m1x6', 'm2_lorma_plus', 'm4_lora', 'h_cat3', 'h_rslora']
ens_top7                       0.5833  0.0  ['m1_rslora', 'm1_lorma_plus', 'm1_full', 'm1_lora_doc_pos', 'kd_student_m1x6', 'm2_lorma_plus', 'm4_lora']
ens_hier                       0.5812  0.0  ['h_cat3', 'h_rslora']
ens_all                        0.5758  0.0  ['m1_rslora', 'm1_lorma_plus', 'm1_full', 'm1_lora_doc_pos', 'kd_student_m1x6', 'm2_lorma_plus', 'm4_lora', 'm1_full_s1', 'm1_lorma_plus_s1', 'm1_rslora_s1', 'h_cat3', 'h_rslora']
ens_top7_seeds                 0.5748  0.0  ['m1_rslora', 'm1_lorma_plus', 'm1_full', 'm1_lora_doc_pos', 'kd_student_m1x6', 'm2_lorma_plus', 'm4_lora', 'm1_full_s1', 'm1_lorma_plus_s1', 'm1_rslora_s1']
h_cat3                         0.5544  0.0  ['h_cat3']
m1_full_s1       

In [7]:
# ---- quick peek: new BiGRU / seed results on dev ----
!grep -h "DONE h_\|DONE m1_.*_s1" /kaggle/working/stats/p3x_gpu0.log /kaggle/working/stats/p3x_gpu1.log

08:13:39 DONE h_rslora: dev macro λ0 0.5415 | λ0.5 0.6019 (best epoch 19) | 1.4 min
08:15:56 DONE h_cat3: dev macro λ0 0.5544 | λ0.5 0.5314 (best epoch 16) | 2.3 min
08:33:07 DONE h_cat3_td | 2.6 min
08:33:33 DONE h_rslora_td | 0.4 min
08:42:15 DONE m1_rslora_s1: dev macro λ0 0.5379 | λ0.5 0.5222 | 9.6 min
08:54:26 DONE m1_full_s1: dev macro λ0 0.5416 | λ0.5 0.5279 | 12.2 min
09:03:47 DONE m1_lorma_plus_s1: dev macro λ0 0.5306 | λ0.5 0.5373 | 9.3 min


In [8]:
# ---- P3-B: back up the new Phase-3 zips + summary ----
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi
HfApi(token=UserSecretsClient().get_secret("HF_WRITE_TOKEN")).upload_folder(
    repo_id="Utkarsh-1718-1/cs781-phase1-work", repo_type="model", folder_path="/kaggle/working",
    commit_message="phase 3 upgrade zips", allow_patterns=["submissions_p3/**", "stats/p3_upgrade.json"])
print("Phase 3 upgrade zips backed up ✅")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Phase 3 upgrade zips backed up ✅
